# ASAP8 DoC dF/F analysis

Compact lab-meeting notebook. Native dF/F traces are shown without baseline subtraction; scalar response comparisons use the stated event windows.

**Plotting convention:** population/trial summaries are shown as mean ± SEM unless otherwise noted.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage
from matplotlib.lines import Line2D
from IPython.display import display, HTML

from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.dataset import DEPTH_GROUP_ORDER, build_voltage_session_table, build_voltage_roi_table
from vip_slap2_analysis.behavior.change_detection import build_change_detection_events
from vip_slap2_analysis.behavior.encoder import compute_encoder_velocity
from vip_slap2_analysis.voltage.responses import load_response_package, get_mean_response, build_single_trial_index

sns.set_style("white")
plt.rcParams.update({"legend.fontsize":"x-large","axes.labelsize":"xx-large","axes.titlesize":"xx-large","xtick.labelsize":"xx-large","ytick.labelsize":"xx-large"})
display(HTML("<style>.container { width:100% !important; }</style>"))


# 1. Setup

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SAVE_PATH = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots")
TARGET_MICE = [852835, 863774]
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check","volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = True
EXPECTED_F0_SMOOTH_SEC = 60.0

SESSION_ORDER = ["A0","A1","A2","B0","B1","B2"]
IMAGE_WINDOW_S = (0.0,0.25)
IMAGE_BASELINE_S = (-0.25,0.0)
IMAGE_CYCLE_S = 0.75
MIN_TRIALS_PER_IMAGE = 5

PEAK_WINDOW_S = (-0.25,0.50)
PEAK_SMOOTH_MS = 10.0
PEAK_MODE = "max"       # "max" = positive maximum; "absolute" = largest |dF/F|
LATENCY_PLOT_WINDOW_S = (-0.25,0.50)

MAX_SEQUENCE_PRESENTATIONS = 12
MIN_EPOCHS_PER_POSITION = 5
MIN_SEQUENCE_POSITIONS = 4
REQUIRE_COMPLETE_SEQUENCE_BLOCK = True
PREFERRED_SLOPE_MODE = "max"  # "max" or "max_abs"

N_MATCHED_CONTROLS = 5
MATCH_POSITION_TOLERANCE = 2
CHANGE_RESPONSE_WINDOW_S = (0.0,0.25)
OMISSION_MATCH_WINDOW_S = (0.0,0.50)
PRE_OMISSION_RAMP_EARLY_S = (-0.750,-0.25)
PRE_OMISSION_RAMP_LATE_S = (-0.25,0.0)
OMISSION_RAMP_EARLY_S = (-0.750,-0.0)
OMISSION_RAMP_LATE_S = (0.0,0.750)
POST_OMISSION_WINDOW_S = (0.75,1.0)
PRE_OMISSION_SLOPE_WINDOW_S = (-0.50,0.75)
OMISSION_RAMP_SLOPE_WINDOW_S = (-0.50,0.75)
WITHIN_SESSION_THIRD_ORDER = ["First third","Second third","Third third"]
# Plot both direct change-vs-previous and matched same-image change modulation by default.
WITHIN_SESSION_CHANGE_METRICS = [
    ("change_minus_pre_dff", "Change − previous image"),
    ("change_minus_matched_same_image_dff", "Change − matched same image"),
]

DEPTH_COLORS = {"<100 µm":"#EBA287","100–150 µm":"#d1e2b0",">150 µm":"#7bbcd5"}

def depth_group_from_um(depth):
    depth=float(depth)
    return "<100 µm" if depth < 100 else ("100–150 µm" if depth <= 150 else ">150 µm")

# --- Exploratory change/omission additions ---
# Keep these controls simple and presentation-oriented. None of the analyses
# below changes the underlying extracted dF/F traces.


RUNNING_KWARGS = dict(
    wheel_radius_cm=4.69,
    encoder_units="ticks",
    ticks_per_revolution=8192,
    absolute_velocity=True,
    time_zero="first_sample",  # source index is HARP time; zero to the first HARP sample to match event/trial times
)
RUNNING_THRESHOLD_CM_S = 5.0
RUNNING_CLASS_WINDOW_S = (-2.0, 2.0)
RUN_RUNNING_CONTROL = True  # lightweight; set False to skip encoder loading

# I do not assume here that A0/A1/A2/B0/B1/B2 map onto
# Familiar/Novel/Novel+ in a particular way. Fill this in once confirmed, e.g.
# EXPERIENCE_STAGE_MAP = {"A0":"Familiar", "B0":"Novel", "B1":"Novel+"}
EXPERIENCE_STAGE_MAP = {}
EXPERIENCE_STAGE_ORDER = ["Familiar", "Novel", "Novel+"]



DEFAULT_MAX_SINGLE_TRIALS = 24
MIN_OMISSION_TRIALS_PER_EXPECTED_IMAGE = 3


## Canonical DoC tables

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)
sessions = build_voltage_session_table(
    registry,subject_ids=TARGET_MICE,paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,trace_variant=TRACE_VARIANT,
    expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC
)
rois = build_voltage_roi_table(
    sessions,registration_filename=REGISTRATION_FILENAME,
    exclude_invalid_rois=EXCLUDE_INVALID_ROIS
)
rois["depth_um"] = pd.to_numeric(rois["depth_um"],errors="coerce")
rois["depth_group"] = rois["depth_um"].map(depth_group_from_um)
events = build_change_detection_events(sessions)
trial_index = build_single_trial_index(sessions,events)

print(f"{len(sessions)} sessions · {rois['included'].sum()} included ROI observations")
display(sessions[["subject_id","session_id","session_label","session_order","dmd1_depth_um","dmd2_depth_um"]])
display(rois.loc[rois["included"],["subject_id","session_label","dmd","roi","depth_um","depth_group"]].drop_duplicates().sort_values(["depth_um","subject_id","session_label"]))


In [ ]:
def finish_axis(ax):
    sns.despine(ax=ax); ax.tick_params(axis="both",labelsize=11)
    for spine in ax.spines.values(): spine.set_linewidth(2)

def add_depth_legend(ax,loc="best"):
    ax.legend(handles=[Line2D([0],[0],color=DEPTH_COLORS[g],lw=3,marker="o",mec="black",mew=.6,label=g) for g in DEPTH_GROUP_ORDER],title="Depth",frameon=False,fontsize=9,loc=loc)

def smooth_finite(y,sigma):
    y=np.asarray(y,float).reshape(-1); good=np.isfinite(y)
    if good.sum()<3: return np.full_like(y,np.nan)
    filled=np.interp(np.arange(y.size),np.flatnonzero(good),y[good])
    return ndimage.gaussian_filter1d(filled,max(0,float(sigma)),mode="nearest")

def decode_strings(values):
    return np.asarray([x.decode() if isinstance(x,(bytes,np.bytes_)) else str(x) for x in np.asarray(values).reshape(-1)])

def roi_number(x):
    if isinstance(x,str):
        hits=re.findall(r"\d+",x)
        if not hits: raise ValueError(f"Could not parse ROI label {x!r}")
        return int(hits[-1])
    return int(x)

def select_roi_example(spec):
    roi=roi_number(spec["roi"])
    q=rois[rois["included"].astype(bool) & rois["subject_id"].astype(str).eq(str(spec["mouse"])) & rois["session_label"].astype(str).eq(str(spec["day"])) & rois["dmd"].astype(int).eq(int(spec["dmd"])) & rois["roi"].astype(int).eq(roi)]
    if q.empty:
        avail=rois[rois["included"].astype(bool) & rois["subject_id"].astype(str).eq(str(spec["mouse"])) & rois["session_label"].astype(str).eq(str(spec["day"]))][["dmd","roi"]].drop_duplicates().values.tolist()
        raise ValueError(f"No included ROI matches {spec}. Available DMD/ROI pairs: {avail}")
    r=q.iloc[0]; s=sessions[sessions["session_id"].astype(str).eq(str(r["session_id"]))].iloc[0]
    return r,s

def source_roi_axis(package,dmd,source_roi):
    ids=decode_strings(package[f"DMD{int(dmd)}"]["roi_ids"]); label=f"DMD{int(dmd)}_ROI{int(source_roi)}"
    hits=np.flatnonzero(ids==label)
    if not len(hits):
        parsed=np.array([int(re.findall(r"\d+",x)[-1]) for x in ids],int); hits=np.flatnonzero(parsed==int(source_roi))
    if not len(hits): raise KeyError(f"{label} is absent from kept ROI axis")
    return int(hits[0])

def h5_roi_axis(group,dmd,source_roi):
    ids=decode_strings(group["roi_ids"][:]); label=f"DMD{int(dmd)}_ROI{int(source_roi)}"; hits=np.flatnonzero(ids==label)
    if not len(hits):
        parsed=np.array([int(re.findall(r"\d+",x)[-1]) for x in ids],int); hits=np.flatnonzero(parsed==int(source_roi))
    if not len(hits): raise KeyError(f"{label} is absent from H5 kept ROI axis")
    return int(hits[0])

def reconcile_timebase(t,n):
    t=np.asarray(t,float).reshape(-1)
    if len(t)==n: return t
    if len(t)<2: raise ValueError(f"Cannot reconcile {len(t)} time samples to {n} trace samples")
    dt=float(np.nanmedian(np.diff(t))); zero=min(int(np.nanargmin(np.abs(t))),n-1)
    return (np.arange(n,dtype=float)-zero)*dt

def window_slice(t,window):
    t=np.asarray(t,float); a,b=map(float,window)
    i0=int(np.searchsorted(t,a,side="left")); i1=int(np.searchsorted(t,b,side="left"))
    if i1<=i0: raise ValueError(f"Window {window} has no samples in timebase [{t[0]}, {t[-1]}]")
    return slice(i0,i1)

def window_mean(y,t,window,axis=-1):
    return np.nanmean(np.asarray(y,float)[...,window_slice(t,window)],axis=axis)

def interp_trace(t,y,grid):
    t=np.asarray(t,float); y=np.asarray(y,float); good=np.isfinite(t)&np.isfinite(y)
    return np.interp(grid,t[good],y[good],left=np.nan,right=np.nan) if good.sum()>=2 else np.full_like(grid,np.nan,dtype=float)


_IMAGE_CONTROL_CACHE={}

def indexed_event_table(session_id,dmd,event_type):
    sid=str(session_id)
    idx=trial_index[trial_index["session_id"].astype(str).eq(sid) & trial_index["dmd"].astype(int).eq(int(dmd)) & trial_index["event_type"].astype(str).eq(str(event_type))].copy()
    if "matched" in idx.columns: idx=idx[idx["matched"].astype(bool)]
    idx["session_id"]=idx["session_id"].astype(str); idx=idx.rename(columns={"onset_sec":"stored_onset_sec","image_name":"stored_image_name"})
    ev=events[events["session_id"].astype(str).eq(sid)].copy(); ev["session_id"]=ev["session_id"].astype(str)
    needed=["session_id","event_id","onset_sec","image_name","image_label","is_change","is_omission","sequence_position_expected","next_event_id"]
    for col in needed:
        if col not in ev: ev[col]=np.nan
    ev=ev[needed].rename(columns={"onset_sec":"cycle_onset_sec","image_name":"event_image_name","image_label":"event_image_label"})
    out=idx.merge(ev,on=["session_id","event_id"],how="left",validate="many_to_one")
    label_map=ev.set_index("event_id")["event_image_label"]; onset_map=ev.set_index("event_id")["cycle_onset_sec"]; pos_map=ev.set_index("event_id")["sequence_position_expected"]
    out["next_image_label"]=out["next_event_id"].map(label_map); out["next_cycle_onset_sec"]=out["next_event_id"].map(onset_map); out["next_sequence_position_expected"]=out["next_event_id"].map(pos_map)
    return out.sort_values("trial_index").reset_index(drop=True)

def load_image_control_table(h5,session_id,dmd,source_roi,axis):
    key=(str(session_id),int(dmd),int(source_roi))
    if key in _IMAGE_CONTROL_CACHE: return _IMAGE_CONTROL_CACHE[key]
    idx=indexed_event_table(session_id,dmd,"image")
    idx=idx[(~idx["is_change"].fillna(False).astype(bool)) & (~idx["is_omission"].fillna(False).astype(bool))].copy()
    stored_t=np.asarray(h5["timebase_sec/image"][:],float); parts=[]
    for path,q in idx.groupby("dataset_path",sort=False):
        q=q.sort_values("trial_index").copy(); rows=q["trial_index"].astype(int).to_numpy(); ds=h5[str(path)]
        t=reconcile_timebase(stored_t,int(ds.shape[-1])); arr=np.asarray(ds[rows,int(axis),:],float)
        q["image_dff"]=window_mean(arr,t,IMAGE_WINDOW_S); q["cycle_dff"]=window_mean(arr,t,OMISSION_MATCH_WINDOW_S); parts.append(q)
    out=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    _IMAGE_CONTROL_CACHE[key]=out
    return out

def nearest_control_rows(label,onset_sec,sequence_position,controls,n_controls=N_MATCHED_CONTROLS,position_tolerance=MATCH_POSITION_TOLERANCE):
    if controls.empty or pd.isna(label): return controls.iloc[0:0]
    candidates=controls[controls["event_image_label"].astype(str).eq(str(label))].copy()
    if np.isfinite(pd.to_numeric(sequence_position,errors="coerce")) and len(candidates):
        pos=pd.to_numeric(candidates["sequence_position_expected"],errors="coerce")
        near=np.abs(pos-float(sequence_position))<=float(position_tolerance)
        if near.any(): candidates=candidates[near]
    if not len(candidates): return candidates
    candidates["match_distance_sec"]=np.abs(pd.to_numeric(candidates["cycle_onset_sec"],errors="coerce")-float(onset_sec))
    return candidates.nsmallest(int(n_controls),"match_distance_sec")

def nearest_control_mean(label,onset_sec,sequence_position,controls,value_col):
    q=nearest_control_rows(label,onset_sec,sequence_position,controls)
    return float(np.nanmean(pd.to_numeric(q[value_col],errors="coerce"))) if len(q) else np.nan

def shifted_window_means(traces,t,offsets,window):
    out=np.full(len(traces),np.nan,float)
    for i,offset in enumerate(np.asarray(offsets,float)):
        if np.isfinite(offset): out[i]=window_mean(traces[i],t,(window[0]+offset,window[1]+offset))
    return out

def shifted_window_slopes(traces,t,offsets,window):
    traces=np.asarray(traces,float); t=np.asarray(t,float); offsets=np.asarray(offsets,float)
    out=np.full(len(traces),np.nan,float)
    for i,offset in enumerate(offsets):
        if not np.isfinite(offset): continue
        sl=window_slice(t,(window[0]+offset,window[1]+offset))
        x=t[sl]-offset; y=traces[i,sl]; good=np.isfinite(x)&np.isfinite(y)
        if good.sum()>=3 and np.nanstd(x[good])>0:
            out[i]=np.polyfit(x[good],y[good],1)[0]
    return out


# 2. Single-neuron raw trial explorer

In [ ]:
# ========================= SELECT EXAMPLE =========================
MOUSE = 852835
SESSION = "A0"          # A0, A1, A2, B0, B1, B2
DMD = 1
ROI = 0
STIM_TYPE = "change"    # "image", "change", or "omission"
IMAGE_NAME = None       # image only: None = all identities; or use an image stem/name
MAX_TRIALS = 30
# =================================================================

spec = dict(mouse=MOUSE, day=SESSION, dmd=DMD, roi=ROI)
r, s = select_roi_example(spec)

with h5py.File(s["single_trial_h5"], "r") as h5:
    group = h5[f"DMD{DMD}"]
    axis = h5_roi_axis(group, DMD, ROI)

    if STIM_TYPE in {"change", "omission"}:
        sub = group[STIM_TYPE]
        traces = np.asarray(sub["traces"][:, axis, :], float)
        t = reconcile_timebase(h5[f"timebase_sec/{STIM_TYPE}"][:], traces.shape[-1])
        trial_labels = np.array([STIM_TYPE] * len(traces))
    elif STIM_TYPE == "image":
        stored_t = np.asarray(h5["timebase_sec/image"][:], float)
        parts, labels = [], []
        for key in group["image_identity"]:
            sub = group["image_identity"][key]
            image_name = str(sub.attrs.get("image_name", key))
            stem = Path(image_name.replace("\\", "/")).stem
            if IMAGE_NAME is not None:
                wanted = Path(str(IMAGE_NAME).replace("\\", "/")).stem
                if stem != wanted:
                    continue
            arr = np.asarray(sub["traces"][:, axis, :], float)
            parts.append(arr)
            labels.extend([stem] * len(arr))
        if not parts:
            raise ValueError(f"No image trials found for IMAGE_NAME={IMAGE_NAME!r}")
        traces = np.concatenate(parts, axis=0)
        t = reconcile_timebase(stored_t, traces.shape[-1])
        trial_labels = np.asarray(labels)
    else:
        raise ValueError("STIM_TYPE must be 'image', 'change', or 'omission'")

take = np.arange(len(traces))
if len(take) > MAX_TRIALS:
    take = np.unique(np.linspace(0, len(traces)-1, MAX_TRIALS).round().astype(int))
shown = traces[take]
mean_trace = np.nanmean(shown, axis=0)

fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.6), gridspec_kw={"width_ratios":[1.2,1]})
c = DEPTH_COLORS[str(r["depth_group"])]

for y in shown:
    axs[0].plot(t, y, color=c, lw=.55, alpha=.20)
axs[0].plot(t, mean_trace, color="black", lw=1.7)

if STIM_TYPE in {"image","change"}:
    axs[0].axvspan(0, .25, color=c, alpha=.10, lw=0)
elif STIM_TYPE == "omission":
    axs[0].axvspan(0, .50, color=".75", alpha=.12, lw=0)
    axs[0].axvline(.75, color=".55", lw=.8, ls=":")
axs[0].axvline(0, color=".3", lw=.9, ls="--")
axs[0].set(xlabel=f"Time from {STIM_TYPE} onset (s)", ylabel="dF/F",
           title=f"{STIM_TYPE.capitalize()} single trials")
finish_axis(axs[0])

finite = shown[np.isfinite(shown)]
vmin, vmax = (np.nanpercentile(finite,[2,98]) if len(finite) else (-1,1))
im = axs[1].imshow(shown, aspect="auto", interpolation="nearest",
                   extent=[t[0],t[-1],len(shown)-.5,-.5], vmin=vmin, vmax=vmax, cmap="viridis")
axs[1].axvline(0, color="white", lw=.8, ls="--")
axs[1].set(xlabel="Time (s)", ylabel="Displayed trial", title="Trial × time")
plt.colorbar(im, ax=axs[1], label="dF/F")

fig.suptitle(
    f'Mouse {r["subject_id"]} · {r["session_label"]} · DMD{int(r["dmd"])} ROI{int(r["roi"])} · '
    f'{r["depth_um"]:.0f} µm · n={len(traces)}',
    fontsize=13, y=1.02
)
fig.tight_layout()
plt.show()

if STIM_TYPE == "image":
    display(pd.Series(trial_labels).value_counts().rename("n_trials").to_frame())

# 3. Image responses
## Peak latency from un-baselined mean dF/F

In [ ]:
selected=sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)]
available_labels=[s for s in SESSION_ORDER if s in set(selected["session_label"].astype(str))]
peak_rows=[]; trace_rows=[]

for session in selected.itertuples(index=False):
    sid=str(session.session_id); pkg=load_response_package(session.mean_npz)
    for r in rois[(rois["session_id"].astype(str)==sid) & rois["included"].astype(bool)].itertuples(index=False):
        dmd,roi,key=int(r.dmd),int(r.roi),f"DMD{int(r.dmd)}"
        if key not in pkg: continue
        image_traces=[]; image_times=[]
        for image in pkg[key].get("image_identity",{}):
            try: t,y=get_mean_response(pkg,dmd=dmd,source_roi=roi,event_type="image",image_name=image)
            except (KeyError,IndexError,ValueError): continue
            t=np.asarray(t,float).reshape(-1); y=np.asarray(y,float).squeeze()
            if y.ndim!=1 or len(y)!=len(t) or len(y)<3: continue
            dt=np.nanmedian(np.diff(t))
            if not np.isfinite(dt) or dt<=0: continue
            ys=smooth_finite(y,(PEAK_SMOOTH_MS/1000)/dt)
            keep=(t>=PEAK_WINDOW_S[0])&(t<=PEAK_WINDOW_S[1])&np.isfinite(ys)
            if not keep.any(): continue
            idx=np.flatnonzero(keep)
            local=np.nanargmax(np.abs(ys[keep])) if PEAK_MODE=="absolute" else np.nanargmax(ys[keep])
            p=idx[local]
            peak_rows.append(dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=dmd,roi=roi,cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group),image_name=str(image),peak_latency_s=float(t[p]),peak_dff=float(ys[p])))
            image_traces.append(y); image_times.append(t)
        if image_traces:
            grid=image_times[0]
            stacked=np.vstack([interp_trace(tt,yy,grid) for tt,yy in zip(image_times,image_traces)])
            trace_rows.append(dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=dmd,roi=roi,cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group),n_images=len(image_traces),time=grid,mean_image_dff=np.nanmean(stacked,axis=0)))

image_peak_latency=pd.DataFrame(peak_rows); mean_image_trace_df=pd.DataFrame(trace_rows)
if image_peak_latency.empty: raise RuntimeError("No image-response peaks extracted.")

keys=["subject_id","session_id","session_label","session_order","cell_id","global_cell_id","manually_registered"]
cell_session_peak=image_peak_latency.groupby(keys,observed=True,dropna=False).agg(peak_latency_s=("peak_latency_s","median"),median_peak_dff=("peak_dff","median"),n_images=("image_name","nunique"),depth_um=("depth_um","median")).reset_index()
cell_session_peak["depth_group"]=cell_session_peak["depth_um"].map(depth_group_from_um)
tracked_peak=cell_session_peak[cell_session_peak["manually_registered"].astype(bool) & cell_session_peak["global_cell_id"].astype(str).ne("") & cell_session_peak["global_cell_id"].astype(str).ne("nan")].groupby(["subject_id","global_cell_id"],group_keys=False,observed=True).filter(lambda x:x["session_id"].nunique()>=2)

xmap={s:i for i,s in enumerate(available_labels)}; x=np.arange(len(available_labels))
fig,ax=plt.subplots(figsize=(5.2,3.6))
for (_,cell),g in tracked_peak.groupby(["subject_id","global_cell_id"],observed=True):
    g=g.assign(x=g["session_label"].map(xmap)).dropna(subset=["x"]).sort_values("x")
    if len(g)>1: ax.plot(g["x"],1000*g["peak_latency_s"],color=DEPTH_COLORS[str(g["depth_group"].iloc[0])],lw=.8,alpha=.16,zorder=1)
for group,g in tracked_peak.groupby("depth_group",observed=True):
    q=g.groupby("session_label")["peak_latency_s"].agg(mean="mean",sem="sem").reindex(available_labels)
    valid=q["mean"].notna().to_numpy(); xx=x[valid]; c=DEPTH_COLORS[str(group)]
    ax.fill_between(xx,1000*(q["mean"]-q["sem"]).to_numpy()[valid],1000*(q["mean"]+q["sem"]).to_numpy()[valid],color=c,alpha=.25,lw=0)
    ax.plot(xx,1000*q["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group),zorder=3)
ax.axhline(0,color=".5",lw=1,ls="--"); ax.axhline(250,color="k",lw=.5,dashes=[6,3],zorder=0)
if "A2" in xmap and "B0" in xmap: ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
ax.set(xticks=x,xticklabels=available_labels,xlabel="Session",ylabel="Time of maximum mean dF/F\nrelative to image onset (ms)",title="Image-response timing across sessions")
finish_axis(ax); ax.legend(title="Depth",frameon=False,fontsize=9); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"latency_shift"),formats=[".pdf"],dpi=300); plt.show()

## Mean image dF/F traces for every neuron across days

In [ ]:
grid=np.linspace(LATENCY_PLOT_WINDOW_S[0],LATENCY_PLOT_WINDOW_S[1],751)
fig,axs=plt.subplots(2,3,figsize=(6.0,5.0),sharex=True,sharey=True); axs=axs.ravel()

for ax,label in zip(axs,SESSION_ORDER):
    q=mean_image_trace_df[mean_image_trace_df["session_label"].astype(str).eq(label)]
    for row in q.itertuples(index=False):
        ax.plot(grid,interp_trace(row.time,row.mean_image_dff,grid),color=DEPTH_COLORS[str(row.depth_group)],lw=1,alpha=.5)
    for group,g in q.groupby("depth_group",observed=True):
        a=np.vstack([interp_trace(row.time,row.mean_image_dff,grid) for row in g.itertuples(index=False)])
        ax.plot(grid,np.nanmedian(a,axis=0),color=DEPTH_COLORS[str(group)],lw=3,label=str(group),zorder=3)
    ax.axvspan(0,IMAGE_WINDOW_S[1],color=".75",alpha=.12,lw=0); ax.axvline(0,color=".45",lw=1,ls="--")
    ax.set_title(label); finish_axis(ax)
for ax in axs[3:]: ax.set_xlabel("Time from image onset (s)")
axs[0].set_ylabel("Mean \u0394F/F$_{0}$"); axs[3].set_ylabel("Mean \u0394F/F$_{0}$"); add_depth_legend(axs[2])
fig.suptitle("Mean image response",y=0.975,fontsize=20)
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"mean_image_dff_all_neurons"),formats=[".pdf"],dpi=300); plt.show()
filen = 'MeanImageResponse'
save_figure(fig,os.path.join(SAVE_PATH,filen),formats=['.pdf'],dpi=300)

## Within-neuron change in the mean image dF/F trace

In [ ]:
pairs=[("A0","A2"),("B0","B2")]
fig,axs=plt.subplots(2,1,figsize=(4.5,5.0),sharex=True,sharey=True)

for ax,(first,last) in zip(axs,pairs):
    diffs=[]
    tracked=mean_image_trace_df[mean_image_trace_df["manually_registered"].astype(bool) & mean_image_trace_df["global_cell_id"].astype(str).ne("") & mean_image_trace_df["global_cell_id"].astype(str).ne("nan")]
    for (mouse,cell),g in tracked.groupby(["subject_id","global_cell_id"],observed=True):
        a=g[g["session_label"].astype(str).eq(first)]; b=g[g["session_label"].astype(str).eq(last)]
        if a.empty or b.empty: continue
        a,b=a.iloc[0],b.iloc[0]; y0=interp_trace(a["time"],a["mean_image_dff"],grid); y1=interp_trace(b["time"],b["mean_image_dff"],grid)
        depth=float(np.nanmedian([a["depth_um"],b["depth_um"]])); group=depth_group_from_um(depth); diff=y1-y0
        diffs.append((group,diff)); ax.plot(grid,diff,color=DEPTH_COLORS[group],lw=.75,alpha=.18)
    for group in DEPTH_GROUP_ORDER:
        arr=[d for g,d in diffs if g==group]
        if arr: ax.plot(grid,np.nanmedian(np.vstack(arr),axis=0),color=DEPTH_COLORS[group],lw=2.8,label=group)
    ax.axhline(0,color=".5",lw=1,ls="--"); ax.axvline(0,color=".45",lw=1,ls="--"); ax.axvspan(0,IMAGE_WINDOW_S[1],color=".75",alpha=.10,lw=0)
    ax.set(xlabel="Time from image onset (s)",title=f"{last} − {first}"); finish_axis(ax)
axs[0].set_ylabel("\u0394 mean \u0394F/F$_{0}$",labelpad=0); add_depth_legend(axs[1])
axs[1].set_ylabel("\u0394 mean \u0394F/F$_{0}$",labelpad=0)
# fig.suptitle("Within-neuron change in mean image response across days",y=0.95,fontsize=18)
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"mean_image_dff_longitudinal_difference"),formats=[".pdf",'.png'],dpi=300); plt.show()

## Trial-wise image variability
Image response = mean dF/F[0–250 ms] − mean dF/F[−250–0 ms].

In [ ]:
metric_keys=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]
trial_rows=[]; metric_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); session_rois=rois[rois["included"].astype(bool) & rois["session_id"].astype(str).eq(sid)]
    with h5py.File(session.single_trial_h5,"r") as h5:
        stored_t=np.asarray(h5["timebase_sec/image"][:],float)
        for r in session_rois.itertuples(index=False):
            group=h5[f"DMD{int(r.dmd)}"]; axis=h5_roi_axis(group,int(r.dmd),int(r.roi))
            ys=[]; labels=[]
            for key in group["image_identity"]:
                sub=group["image_identity"][key]; n=int(sub["traces"].shape[0])
                if n<MIN_TRIALS_PER_IMAGE: continue
                t=reconcile_timebase(stored_t,int(sub["traces"].shape[-1]))
                base=np.nanmean(np.asarray(sub["traces"][:,axis,window_slice(t,IMAGE_BASELINE_S)],float),axis=1)
                resp=np.nanmean(np.asarray(sub["traces"][:,axis,window_slice(t,IMAGE_WINDOW_S)],float),axis=1)
                image=str(sub.attrs.get("image_name",key)); delta=resp-base
                ys.append(delta); labels.extend([image]*len(delta))
            if not ys: continue
            y=np.concatenate(ys); labels=np.asarray(labels); good=np.isfinite(y); y=y[good]; labels=labels[good]
            if len(y)<2 or len(np.unique(labels))<2: continue
            grand=float(np.mean(y)); total=float(np.mean((y-grand)**2))
            stats=pd.DataFrame({"image":labels,"y":y}).groupby("image")["y"].agg(["mean","size"])
            between=float(np.sum(stats["size"]*(stats["mean"]-grand)**2)/len(y))
            meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
            metric_rows.append({**meta,"mean_image_delta_dff":grand,"total_dff_variance":total,"identity_dff_variance":between,"image_rms_dff":float(np.sqrt(between)),"image_fve":float(between/total) if total>0 else np.nan,"n_trials":len(y),"n_images":len(stats)})
            trial_rows.append(pd.DataFrame({**{k:[v]*len(y) for k,v in meta.items()},"image_label":labels,"image_delta_dff":y}))

image_trial_df=pd.concat(trial_rows,ignore_index=True); image_metrics=pd.DataFrame(metric_rows)
registered_metrics=image_metrics[image_metrics["manually_registered"].astype(bool) & image_metrics["global_cell_id"].astype(str).ne("") & image_metrics["global_cell_id"].astype(str).ne("nan")].copy()
cell_metrics=registered_metrics.groupby(["subject_id","global_cell_id","depth_group"],observed=True).agg(depth_um=("depth_um","median"),total_dff_variance=("total_dff_variance","median"),image_fve=("image_fve","median"),image_rms_dff=("image_rms_dff","median"),n_sessions=("session_id","nunique")).reset_index()
display(image_metrics.head())

In [ ]:
# Aggregate variance across neurons
order=[g for g in DEPTH_GROUP_ORDER if g in set(cell_metrics["depth_group"].astype(str))]
fig,axs=plt.subplots(1,2,figsize=(9.2,3.6))

sns.violinplot(data=cell_metrics,x="depth_group",y="total_dff_variance",order=order,palette=DEPTH_COLORS,
               inner=None,width=.55,linewidth=1.2,ax=axs[0])
rng=np.random.default_rng(8)
xpos=np.array([order.index(str(x)) for x in cell_metrics["depth_group"]],float)+rng.uniform(-.12,.12,len(cell_metrics))
axs[0].scatter(xpos,cell_metrics["total_dff_variance"],s=28,c=[DEPTH_COLORS[str(x)] for x in cell_metrics["depth_group"]],
               edgecolor="black",linewidth=.45,zorder=3,alpha=.85)
axs[0].set(xlabel="Depth",ylabel="Trial-wise ΔF/F variance",title="Total image-response variability")
finish_axis(axs[0])

# Session-wise variance
labels=[s for s in SESSION_ORDER if s in set(image_metrics["session_label"].astype(str))]
x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}
for group,g in image_metrics.groupby("depth_group",observed=True):
    q=(g.groupby("session_label",observed=True)["total_dff_variance"]
       .agg(mean="mean",sem="sem")
       .reindex(labels))
    valid=q["mean"].notna().to_numpy()
    if not valid.any(): continue
    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    axs[1].fill_between(xx,(q["mean"]-q["sem"]).to_numpy()[valid],(q["mean"]+q["sem"]).to_numpy()[valid],color=c,alpha=.12,lw=0)
    axs[1].plot(xx,q["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))
if "A2" in xmap and "B0" in xmap: axs[1].axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
axs[1].set(xticks=x,xticklabels=labels,xlabel="Session",ylabel="Trial-wise ΔF/F variance",title="Across sessions")
finish_axis(axs[1]); add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"image_trial_dff_variance"),formats=[".pdf"],dpi=300); plt.show()

# FVE and RMS
fig,axs=plt.subplots(1,2,figsize=(9.2,3.6))
for ax,metric,ylabel,title in [
    (axs[0],"image_fve","FVE","FVE by image identity"),
    (axs[1],"image_rms_dff","RMS mod. (ΔF/F)","Image-identity response magnitude"),
]:
    for group,g in image_metrics.groupby("depth_group",observed=True):
        q=(g.groupby("session_label",observed=True)[metric]
           .agg(mean="mean",sem="sem")
           .reindex(labels))
        valid=q["mean"].notna().to_numpy()
        if not valid.any(): continue
        xx=x[valid]; c=DEPTH_COLORS[str(group)]
        ax.fill_between(xx,(q["mean"]-q["sem"]).to_numpy()[valid],(q["mean"]+q["sem"]).to_numpy()[valid],color=c,alpha=.12,lw=0)
        ax.plot(xx,q["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))
    ax.axhline(0,color=".6",lw=1,ls="--")
    ax.set(xticks=x,xticklabels=labels,xlabel="Session",ylabel=ylabel,title=title)
    finish_axis(ax)
add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"image_identity_dff_fve_rms"),formats=[".pdf"],dpi=300); plt.show()

# 4. Image-sequence dynamics
Sequence-position responses use the same 0–250 ms minus −250–0 ms image metric.

In [ ]:
seq_pos_rows=[]; seq_slope_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); pkg=load_response_package(session.sequence_npz); t0=np.asarray(pkg["timebase_sec"]["image"],float)
    for r in rois[rois["included"].astype(bool) & rois["session_id"].astype(str).eq(sid)].itertuples(index=False):
        dmd=int(r.dmd); roi=int(r.roi); key=f"DMD{dmd}"
        if key not in pkg: continue
        axis=source_roi_axis(pkg,dmd,roi)
        meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=dmd,roi=roi,cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
        for image,entry in pkg[key].get("image_identity",{}).items():
            rep=entry.get("repeated",{})
            if not rep or "mean" not in rep: continue
            traces=np.asarray(rep["mean"],float)[:,axis,:]; pos=np.asarray(rep["positions"],int); counts=np.asarray(rep["counts"],int)
            t=reconcile_timebase(t0,traces.shape[-1]); delta=window_mean(traces,t,IMAGE_WINDOW_S)-window_mean(traces,t,IMAGE_BASELINE_S)
            keep=(pos<=MAX_SEQUENCE_PRESENTATIONS)&(counts>=MIN_EPOCHS_PER_POSITION)&np.isfinite(delta)
            p,y,w=pos[keep].astype(float),delta[keep].astype(float),counts[keep].astype(float)
            for pp,yy,ww in zip(p,y,w): seq_pos_rows.append({**meta,"sequence_image":str(image),"sequence_position":int(pp),"mean_delta_dff":float(yy),"n_epochs":int(ww)})
            if len(p)<MIN_SEQUENCE_POSITIONS or len(np.unique(p))<MIN_SEQUENCE_POSITIONS: continue
            slope,intercept=np.polyfit(p,y,1,w=np.sqrt(w)); pred=intercept+slope*p; ss_res=np.sum(w*(y-pred)**2); ybar=np.average(y,weights=w); ss_tot=np.sum(w*(y-ybar)**2)
            seq_slope_rows.append({**meta,"sequence_image":str(image),"sequence_slope_dff_per_presentation":float(slope),"sequence_intercept_dff":float(intercept),"sequence_r2":float(1-ss_res/ss_tot) if ss_tot>0 else np.nan,"n_positions":len(p),"min_epochs_per_position":int(w.min())})

sequence_position_df=pd.DataFrame(seq_pos_rows); sequence_slopes=pd.DataFrame(seq_slope_rows)

# One pooled slope per neuron/session, with image identity treated as a fixed intercept.
neuron_rows=[]
group_keys=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]
for keys,g in sequence_position_df.groupby(group_keys,observed=True,dropna=False):
    if g["sequence_image"].nunique()<2: continue
    pieces=[]
    for image,h in g.groupby("sequence_image",observed=True):
        x=h["sequence_position"].to_numpy(float); y=h["mean_delta_dff"].to_numpy(float); w=h["n_epochs"].to_numpy(float)
        if len(np.unique(x))<2: continue
        xc=x-np.average(x,weights=w); yc=y-np.average(y,weights=w)
        pieces.append((xc,yc,w))
    if not pieces: continue
    xc=np.concatenate([p[0] for p in pieces]); yc=np.concatenate([p[1] for p in pieces]); w=np.concatenate([p[2] for p in pieces])
    denom=np.sum(w*xc**2)
    if denom<=0: continue
    row=dict(zip(group_keys,keys)); row["sequence_slope_dff_per_presentation"]=float(np.sum(w*xc*yc)/denom); row["n_images"]=g["sequence_image"].nunique(); row["n_position_means"]=len(g)
    neuron_rows.append(row)

neuron_sequence_slopes=pd.DataFrame(neuron_rows)
print(f"{len(sequence_slopes):,} image-specific slopes · {len(neuron_sequence_slopes):,} neuron/session pooled slopes")
display(neuron_sequence_slopes.head())

## Selectable sequence example

In [ ]:
SEQUENCE_EXAMPLE = dict(mouse=863774, day="A2", dmd=1, roi=0, image=None)

r, s = select_roi_example(SEQUENCE_EXAMPLE)
sid = str(r["session_id"])
dmd = int(r["dmd"])
roi = int(r["roi"])

q = sequence_slopes[
    sequence_slopes["session_id"].astype(str).eq(sid)
    & sequence_slopes["dmd"].eq(dmd)
    & sequence_slopes["roi"].eq(roi)
].copy()

if q.empty:
    raise ValueError(f"No sequence slopes available for {SEQUENCE_EXAMPLE}")

if SEQUENCE_EXAMPLE["image"] is None:
    ex = q.loc[q["sequence_slope_dff_per_presentation"].abs().idxmax()]
else:
    stem = Path(str(SEQUENCE_EXAMPLE["image"]).replace("\\", "/")).stem
    hit = q[
        q["sequence_image"].astype(str)
        .map(lambda x: Path(x.replace("\\", "/")).stem)
        .eq(stem)
    ]
    if hit.empty:
        raise ValueError(
            f"Image {SEQUENCE_EXAMPLE['image']!r} not found. "
            f"Available: {q['sequence_image'].unique().tolist()}"
        )
    ex = hit.iloc[0]

# ---------- Load selected sequence ----------
pkg = load_response_package(s["sequence_npz"])
dpkg = pkg[f"DMD{dmd}"]
stem = Path(str(ex["sequence_image"]).replace("\\", "/")).stem

ikey = next(
    (
        k for k in dpkg.get("image_identity", {})
        if Path(str(k).replace("\\", "/")).stem == stem
    ),
    None,
)
if ikey is None:
    raise KeyError(f"{stem!r} not found in {s['sequence_npz']}")

rep = dpkg["image_identity"][ikey]["repeated"]
axis = source_roi_axis(pkg, dmd, roi)

traces = np.asarray(rep["mean"], float)[:, axis, :]
std_traces = (
    np.asarray(rep["std"], float)[:, axis, :]
    if "std" in rep else None
)
n_finite = (
    np.asarray(rep["n_finite"], float)[:, axis, :]
    if "n_finite" in rep else None
)

pos = np.asarray(rep["positions"], int)
counts = np.asarray(rep["counts"], int)

t = reconcile_timebase(
    pkg["timebase_sec"]["image"],
    traces.shape[-1],
)

response = window_mean(traces, t, IMAGE_WINDOW_S)
baseline = window_mean(traces, t, IMAGE_BASELINE_S)
delta = response - baseline

keep = (
    (pos <= MAX_SEQUENCE_PRESENTATIONS)
    & (counts >= MIN_EPOCHS_PER_POSITION)
    & np.isfinite(delta)
)

pos = pos[keep]
counts = counts[keep]
traces = traces[keep]
delta = delta[keep]

if std_traces is not None:
    std_traces = std_traces[keep]
if n_finite is not None:
    n_finite = n_finite[keep]

order = np.argsort(pos)

pos = pos[order]
counts = counts[order]
traces = traces[order]
delta = delta[order]

if std_traces is not None:
    std_traces = std_traces[order]
if n_finite is not None:
    n_finite = n_finite[order]

c = DEPTH_COLORS[str(r["depth_group"])]

# ---------- SEM for ΔdF/F ----------
delta_sem = None

if std_traces is not None:
    if n_finite is None:
        n_finite = np.broadcast_to(
            counts[:, None],
            std_traces.shape,
        ).astype(float)

    point_sem = std_traces / np.sqrt(
        np.maximum(n_finite, 1)
    )

    resp_sem = np.nanmean(
        point_sem[:, window_slice(t, IMAGE_WINDOW_S)],
        axis=1,
    )
    base_sem = np.nanmean(
        point_sem[:, window_slice(t, IMAGE_BASELINE_S)],
        axis=1,
    )

    delta_sem = np.sqrt(
        resp_sem**2 + base_sem**2
    )

# ============================================================
# 1. Sequence-position response + linear adaptation fit
# ============================================================
fig, ax = plt.subplots(figsize=(4.7, 3.6))

ax.errorbar(
    pos,
    delta,
    yerr=delta_sem,
    fmt="-o",
    color=c,
    lw=2.2,
    ms=6,
    mec="black",
    mew=0.7,
    ecolor=c,
    elinewidth=1,
    capsize=2.5,
    zorder=3,
)

xx = np.linspace(
    pos.min(),
    pos.max(),
    200,
)

ax.plot(
    xx,
    ex["sequence_intercept_dff"]
    + ex["sequence_slope_dff_per_presentation"] * xx,
    color="black",
    lw=2,
    ls="--",
    label="Linear fit",
    zorder=4,
)

ax.set(
    xlabel="Presentation after image change",
    ylabel="Image response (ΔdF/F)",
    title=(
        f'Mouse {r["subject_id"]} · {r["session_label"]} · '
        f'DMD{dmd} ROI{roi}\n'
        f'{stem} · slope='
        f'{ex["sequence_slope_dff_per_presentation"]:.4f} '
        f'ΔdF/F/presentation'
    ),
)

ax.set_xticks(pos)

finish_axis(ax)
ax.legend(frameon=False, fontsize=8)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_example_dff_response",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()


# ============================================================
# 2. Contiguous raw dF/F sequence
#
# Keep the FRONT of each aligned trace intact.
# Only truncate the BACK at +IMAGE_CYCLE_S.
# Then concatenate chunks end-to-end and recompute image-onset
# positions in the resulting time axis.
# ============================================================

dt = float(np.nanmedian(np.diff(t)))

# Back-end truncation only.
chunk_mask = t < IMAGE_CYCLE_S
chunk_t = t[chunk_mask]

concat_y = []
image_onsets = []

cursor = 0.0

for y in traces:

    y_chunk = y[chunk_mask]

    # Image onset is wherever t=0 falls within this chunk.
    onset_offset = -chunk_t[0]

    image_onsets.append(
        cursor + onset_offset
    )

    concat_y.append(y_chunk)

    # Advance cursor by exactly the duration of this truncated trace.
    cursor += len(y_chunk) * dt

concat_y = np.concatenate(concat_y)

# Continuous plotting timebase.
concat_t = np.arange(
    len(concat_y),
    dtype=float,
) * dt

# Put the first sample at zero for a simple contiguous axis.
image_onsets = np.asarray(image_onsets)

fig, ax = plt.subplots(
    figsize=(9.0, 3.4)
)

ax.plot(
    concat_t,
    concat_y,
    color="black",
    lw=1.1,
)

# Image windows at their NEW positions in the concatenated trace.
for onset in image_onsets:

    ax.axvspan(
        onset,
        onset + IMAGE_WINDOW_S[1],
        color=c,
        alpha=0.10,
        lw=0,
        zorder=0,
    )

    ax.axvline(
        onset,
        color="0.75",
        lw=0.5,
        zorder=0,
    )

ax.set(
    xlabel="Time through concatenated sequence (s)",
    ylabel="Mean dF/F",
    title=(
        f'Sequence dF/F · Mouse {r["subject_id"]} '
        f'{r["session_label"]} · DMD{dmd} ROI{roi} · {stem}'
    ),
)

finish_axis(ax)
fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_example_raw_dff",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()

In [ ]:
# Compare facilitating vs adapting sequence examples on common axes

SEQUENCE_COMPARE = {
    "Adapting": dict(mouse=852835, day="A2", dmd=1, roi=2, image=None),
    "Facilitating":     dict(mouse=852835, day="A1", dmd=2, roi=0, image=None),
}

COMPARE_COLORS = {
    "Adapting": "#4C9BD5",
    "Facilitating": "#E6866A",
}


def load_sequence_comparison(spec, mode):
    r, s = select_roi_example(spec)

    sid = str(r["session_id"])
    dmd = int(r["dmd"])
    roi = int(r["roi"])

    q = sequence_slopes[
        sequence_slopes["session_id"].astype(str).eq(sid)
        & sequence_slopes["dmd"].eq(dmd)
        & sequence_slopes["roi"].eq(roi)
    ].copy()

    if q.empty:
        raise ValueError(f"No sequence slopes available for {spec}")

    if spec["image"] is None:
        if mode == "Facilitating":
            ex = q.loc[q["sequence_slope_dff_per_presentation"].idxmax()]
        else:
            ex = q.loc[q["sequence_slope_dff_per_presentation"].idxmin()]
    else:
        stem = Path(str(spec["image"]).replace("\\", "/")).stem
        hit = q[
            q["sequence_image"].astype(str)
            .map(lambda x: Path(x.replace("\\", "/")).stem)
            .eq(stem)
        ]

        if hit.empty:
            raise ValueError(
                f"Image {spec['image']!r} not found. "
                f"Available: {q['sequence_image'].unique().tolist()}"
            )

        ex = hit.iloc[0]

    # ---------- Sequence data ----------
    pkg = load_response_package(s["sequence_npz"])
    dpkg = pkg[f"DMD{dmd}"]

    stem = Path(
        str(ex["sequence_image"]).replace("\\", "/")
    ).stem

    ikey = next(
        (
            k for k in dpkg.get("image_identity", {})
            if Path(str(k).replace("\\", "/")).stem == stem
        ),
        None,
    )

    if ikey is None:
        raise KeyError(
            f"{stem!r} not found in {s['sequence_npz']}"
        )

    rep = dpkg["image_identity"][ikey]["repeated"]
    axis = source_roi_axis(pkg, dmd, roi)

    traces = np.asarray(
        rep["mean"],
        float,
    )[:, axis, :]

    std_traces = (
        np.asarray(rep["std"], float)[:, axis, :]
        if "std" in rep
        else None
    )

    n_finite = (
        np.asarray(rep["n_finite"], float)[:, axis, :]
        if "n_finite" in rep
        else None
    )

    pos = np.asarray(
        rep["positions"],
        int,
    )

    counts = np.asarray(
        rep["counts"],
        int,
    )

    t = reconcile_timebase(
        pkg["timebase_sec"]["image"],
        traces.shape[-1],
    )

    # ---------- Response magnitude ----------
    response = window_mean(
        traces,
        t,
        IMAGE_WINDOW_S,
    )

    baseline = window_mean(
        traces,
        t,
        IMAGE_BASELINE_S,
    )

    delta = response - baseline

    keep = (
        (pos <= MAX_SEQUENCE_PRESENTATIONS)
        & (counts >= MIN_EPOCHS_PER_POSITION)
        & np.isfinite(delta)
    )

    pos = pos[keep]
    counts = counts[keep]
    traces = traces[keep]
    delta = delta[keep]

    if std_traces is not None:
        std_traces = std_traces[keep]

    if n_finite is not None:
        n_finite = n_finite[keep]

    order = np.argsort(pos)

    pos = pos[order]
    counts = counts[order]
    traces = traces[order]
    delta = delta[order]

    if std_traces is not None:
        std_traces = std_traces[order]

    if n_finite is not None:
        n_finite = n_finite[order]

    # ---------- Approximate ΔdF/F SEM ----------
    delta_sem = None

    if std_traces is not None:

        if n_finite is None:
            n_finite = np.broadcast_to(
                counts[:, None],
                std_traces.shape,
            ).astype(float)

        point_sem = (
            std_traces
            / np.sqrt(np.maximum(n_finite, 1))
        )

        response_sem = np.nanmean(
            point_sem[
                :,
                window_slice(
                    t,
                    IMAGE_WINDOW_S,
                )
            ],
            axis=1,
        )

        baseline_sem = np.nanmean(
            point_sem[
                :,
                window_slice(
                    t,
                    IMAGE_BASELINE_S,
                )
            ],
            axis=1,
        )

        delta_sem = np.sqrt(
            response_sem**2
            + baseline_sem**2
        )

    # ---------- Contiguous raw sequence ----------
    # Keep the entire FRONT of each event-aligned trace.
    # Truncate only the back end at +750 ms.
    dt = float(
        np.nanmedian(np.diff(t))
    )

    chunk_mask = t < IMAGE_CYCLE_S
    chunk_t = t[chunk_mask]

    concat_y = []
    image_onsets = []

    cursor = 0.0

    for y in traces:

        y_chunk = y[chunk_mask]

        # Position of t=0 inside this retained chunk.
        onset_offset = -chunk_t[0]

        image_onsets.append(
            cursor + onset_offset
        )

        concat_y.append(
            y_chunk
        )

        cursor += len(y_chunk) * dt

    concat_y = np.concatenate(
        concat_y
    )

    concat_t = (
        np.arange(
            len(concat_y),
            dtype=float,
        )
        * dt
    )

    return {
        "label": mode,
        "r": r,
        "s": s,
        "ex": ex,
        "stem": stem,
        "pos": pos,
        "delta": delta,
        "delta_sem": delta_sem,
        "concat_t": concat_t,
        "concat_y": concat_y,
        "image_onsets": np.asarray(image_onsets),
    }


examples = {
    label: load_sequence_comparison(
        spec,
        label,
    )
    for label, spec in SEQUENCE_COMPARE.items()
}


# ============================================================
# 1. Facilitating vs adapting response slopes
# ============================================================

fig, ax = plt.subplots(
    figsize=(5.5, 3.75)
)

for label, d in examples.items():

    color = COMPARE_COLORS[label]

    ax.errorbar(
        d["pos"],
        d["delta"],
        yerr=d["delta_sem"],
        fmt="-o",
        color=color,
        lw=2.2,
        ms=6,
        mec="black",
        mew=0.7,
        ecolor=color,
        elinewidth=1,
        capsize=2.5,
        label=(
            f'{label} '
            f'({d["ex"]["sequence_slope_dff_per_presentation"]:.4f})'
        ),
        zorder=3,
    )

    xx = np.linspace(
        d["pos"].min(),
        d["pos"].max(),
        200,
    )

    ax.plot(
        xx,
        d["ex"]["sequence_intercept_dff"]
        + d["ex"]["sequence_slope_dff_per_presentation"] * xx,
        color=color,
        lw=1.8,
        ls="--",
        alpha=0.9,
        zorder=2,
    )


ax.axhline(
    0,
    color="0.6",
    lw=1,
    ls=":",
)

ax.set(
    xlabel="Presentation after image change",
    ylabel="Image response (ΔdF/F)",
    title="Facilitating vs adapting sequence responses",
)

finish_axis(ax)

ax.legend(
    frameon=False,
    fontsize=8,
)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_facilitating_vs_adapting_response",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()


# ============================================================
# 2. Facilitating vs adapting contiguous raw dF/F sequences
# ============================================================

fig, ax = plt.subplots(
    figsize=(5.5, 3.75)
)

# Draw image epochs once using the first example.
reference = next(
    iter(examples.values())
)

for onset in reference["image_onsets"]:

    ax.axvspan(
        onset,
        onset + IMAGE_WINDOW_S[1],
        color="0.75",
        alpha=0.12,
        lw=0,
        zorder=0,
    )

    ax.axvline(
        onset,
        color="0.82",
        lw=0.5,
        zorder=0,
    )


for label, d in examples.items():

    ax.plot(
        d["concat_t"],
        d["concat_y"],
        color=COMPARE_COLORS[label],
        lw=1.25,
        alpha=0.95,
        label=(
            f'{label}: '
            f'Mouse {d["r"]["subject_id"]} '
            f'{d["r"]["session_label"]} · '
            f'DMD{int(d["r"]["dmd"])} '
            f'ROI{int(d["r"]["roi"])} · '
            f'{d["stem"]}'
        ),
    )


ax.set(
    xlabel="Time through concatenated sequence (s)",
    ylabel="Mean dF/F",
    title="Facilitating vs adapting sequence dF/F",
)

finish_axis(ax)

ax.legend(
    frameon=False,
    fontsize=7,
)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_facilitating_vs_adapting_raw_dff",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()

## Sequence slope by session and depth

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.8))

xmap = {s: i for i, s in enumerate(SESSION_ORDER)}
x = np.arange(len(SESSION_ORDER))

for group, g in neuron_sequence_slopes.groupby("depth_group", observed=True):
    c = DEPTH_COLORS[str(group)]

    q = (
        g.groupby("session_label")["sequence_slope_dff_per_presentation"]
        .agg(mean="mean", sem="sem")
        .reindex(SESSION_ORDER)
    )

    valid = q["mean"].notna().to_numpy()
    xx = x[valid]

    ax.fill_between(
        xx,
        (q["mean"] - q["sem"]).to_numpy()[valid],
        (q["mean"] + q["sem"]).to_numpy()[valid],
        color=c,
        alpha=0.12,
        lw=0,
        zorder=1,
    )

    ax.plot(
        xx,
        q["mean"].to_numpy()[valid],
        "-o",
        color=c,
        lw=2.8,
        ms=6,
        mec="black",
        mew=0.7,
        label=str(group),
        zorder=2,
    )

ax.axhline(
    0,
    color=".55",
    lw=1,
    ls="--",
)

ax.axvline(
    2.5,
    color=".7",
    lw=1,
    ls=":",
)

ax.set(
    xticks=x,
    xticklabels=SESSION_ORDER,
    xlabel="Session",
    ylabel="Sequence slope\n(\u0394F/F$_{0}$ / presentation)",
    title="Image-sequence slope across sessions",
)

finish_axis(ax)
add_depth_legend(ax)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_dff_slopes_by_depth",
    ),
    formats=[".pdf",'.png'],
    dpi=300,
)

plt.show()

## Longitudinal change in each neuron's strongest image-sequence slope

In [ ]:
def preferred_sequence_block(df,labels):
    start=labels[0]; q=df[df["session_label"].astype(str).isin(labels)&df["manually_registered"].astype(bool)&df["global_cell_id"].astype(str).ne("")&df["global_cell_id"].astype(str).ne("nan")].copy()
    s=q[q["session_label"].astype(str).eq(start)].copy(); s["rank_value"]=s["sequence_slope_dff_per_presentation"].abs() if PREFERRED_SLOPE_MODE=="max_abs" else s["sequence_slope_dff_per_presentation"]
    pref=s.sort_values("rank_value").drop_duplicates(["subject_id","global_cell_id"],keep="last")[["subject_id","global_cell_id","sequence_image","sequence_slope_dff_per_presentation","depth_group"]].rename(columns={"sequence_image":"preferred_image","sequence_slope_dff_per_presentation":"start_slope","depth_group":"start_depth_group"})
    q=q.merge(pref,on=["subject_id","global_cell_id"],how="inner"); q=q[q["sequence_image"].eq(q["preferred_image"])]
    if REQUIRE_COMPLETE_SEQUENCE_BLOCK:
        complete=q.groupby(["subject_id","global_cell_id"],observed=True)["session_label"].nunique(); keep=complete[complete==len(labels)].index
        q=q.set_index(["subject_id","global_cell_id"]).loc[keep].reset_index() if len(keep) else q.iloc[0:0]
    q["delta_sequence_slope"]=q["sequence_slope_dff_per_presentation"]-q["start_slope"]; q["block"]=f"{labels[0]}→{labels[-1]}"
    return q

preferred_sequence_change=pd.concat([preferred_sequence_block(sequence_slopes,["A0","A1","A2"]),preferred_sequence_block(sequence_slopes,["B0","B1","B2"])],ignore_index=True)
blocks=[("A0→A2",["A0","A1","A2"]),("B0→B2",["B0","B1","B2"])]
fig,axs=plt.subplots(1,2,figsize=(8.4,3.5),sharey=True)
for ax,(block,labels) in zip(axs,blocks):
    q=preferred_sequence_change[preferred_sequence_change["block"].eq(block)].copy(); xm={s:i for i,s in enumerate(labels)}
    for (_,cell),g in q.groupby(["subject_id","global_cell_id"],observed=True):
        g=g.assign(x=g["session_label"].map(xm)).dropna(subset=["x"]).sort_values("x"); color=DEPTH_COLORS[str(g["start_depth_group"].iloc[0])]
#         ax.plot(g["x"],g["delta_sequence_slope"],"-o",color=color,lw=1,alpha=.22,ms=4)
    for group,g in q.groupby("start_depth_group",observed=True):
        s=g.groupby("session_label")["delta_sequence_slope"].agg(mean="mean",sem="sem").reindex(labels)
        valid=s["mean"].notna().to_numpy(); xx=np.arange(len(labels))[valid]; color=DEPTH_COLORS[str(group)]
        ax.fill_between(xx,(s["mean"]-s["sem"]).to_numpy()[valid],(s["mean"]+s["sem"]).to_numpy()[valid],color=color,alpha=.12,lw=0)
        ax.plot(xx,s["mean"].to_numpy()[valid],"-o",color=color,lw=3,ms=7,mec="black",mew=.8,label=str(group),zorder=3)
    ax.axhline(0,color=".55",lw=1,ls="--"); ax.set(xticks=np.arange(len(labels)),xticklabels=labels,xlabel="Session",title=block); finish_axis(ax)
axs[0].set_ylabel("Δ preferred-image sequence slope\nfrom first day (ΔdF/F / presentation)"); add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"preferred_sequence_dff_slope_change"),formats=[".pdf"],dpi=300); plt.show()

# 5. Image-change responses
Native traces are un-baselined. Scalar metrics compare the change window with the previous image or matched ordinary presentations of the same image.

In [ ]:
change_rows=[]; change_event_rows=[]; change_curve_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); session_rois=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(sid)]
    with h5py.File(session.single_trial_h5,"r") as h5:
        stored_t=np.asarray(h5["timebase_sec/change"][:],float)
        for r in session_rois.itertuples(index=False):
            group=h5[f"DMD{int(r.dmd)}"]; axis=h5_roi_axis(group,int(r.dmd),int(r.roi)); sub=group["change"]
            traces=np.asarray(sub["traces"][:,axis,:],float); t=reconcile_timebase(stored_t,traces.shape[-1]); idx=indexed_event_table(sid,int(r.dmd),"change")
            if len(idx)!=len(traces) or not np.array_equal(idx["trial_index"].astype(int).to_numpy(),np.arange(len(traces))): raise ValueError(f"Change trial index mismatch for {sid} DMD{int(r.dmd)}")
            offsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()-pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy()
            pre=shifted_window_means(traces,t,offsets,(-.75,-.50)); change=shifted_window_means(traces,t,offsets,CHANGE_RESPONSE_WINDOW_S)
            controls=load_image_control_table(h5,sid,int(r.dmd),int(r.roi),axis)
            matched=np.array([nearest_control_mean(row.event_image_label,row.cycle_onset_sec,row.sequence_position_expected,controls,"image_dff") for row in idx.itertuples(index=False)],float)
            meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
            change_rows.append({**meta,"pre_change_dff":float(np.nanmean(pre)),"change_dff":float(np.nanmean(change)),"matched_same_image_dff":float(np.nanmean(matched)),"change_minus_pre_dff":float(np.nanmean(change-pre)),"change_minus_matched_same_image_dff":float(np.nanmean(change-matched)),"n_changes":len(traces)})
            change_event_rows.append(pd.DataFrame({**{k:[v]*len(traces) for k,v in meta.items()},"event_id":idx["event_id"].to_numpy(),"cycle_onset_sec":pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy(),"image_label":idx["event_image_label"].astype(str).to_numpy(),"pre_change_dff":pre,"change_dff":change,"change_minus_pre_dff":change-pre,"matched_same_image_dff":matched,"change_minus_matched_same_image_dff":change-matched}))
            change_curve_rows.append({**meta,"time":t,"mean_dff":np.nanmean(traces,axis=0),"n_changes":len(traces)})

change_metrics=pd.DataFrame(change_rows); change_event_df=pd.concat(change_event_rows,ignore_index=True); change_curve_df=pd.DataFrame(change_curve_rows)

# Assign each change event to an equal-duration temporal third of the full session.
_bounds=(events.assign(session_id=events["session_id"].astype(str),event_onset_sec=pd.to_numeric(events["onset_sec"],errors="coerce"))
         .groupby("session_id",observed=True)["event_onset_sec"].agg(session_start_sec="min",session_stop_sec="max").reset_index())
change_event_df=change_event_df.merge(_bounds,on="session_id",how="left",validate="many_to_one")
_duration=change_event_df["session_stop_sec"]-change_event_df["session_start_sec"]
change_event_df["session_fraction"]=(change_event_df["cycle_onset_sec"]-change_event_df["session_start_sec"])/_duration.where(_duration>0,np.nan)
change_event_df["session_fraction"]=change_event_df["session_fraction"].clip(0,1)
change_event_df["session_third"]=pd.cut(change_event_df["session_fraction"],[-np.inf,1/3,2/3,np.inf],labels=WITHIN_SESSION_THIRD_ORDER,ordered=True)

change_third_metrics=(change_event_df.dropna(subset=["session_third","change_minus_pre_dff"])
    .groupby(["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","depth_um","depth_group","session_third"],observed=True)
    .agg(
        change_minus_pre_dff=("change_minus_pre_dff","mean"),
        change_minus_matched_same_image_dff=("change_minus_matched_same_image_dff","mean"),
        change_dff=("change_dff","mean"),
        n_change_events=("event_id","count"),
    )
    .reset_index())

display(change_metrics.head())
display(change_third_metrics.head())

## Single-trial change examples by depth

In [ ]:
# ==================== SELECT CHANGE EXAMPLES =====================
day = 'A1'

CHANGE_EXAMPLES = {
    "<100 µm":    dict(mouse=852835, day=day, dmd=1, roi=0),
    "100–150 µm": dict(mouse=852835, day=day, dmd=2, roi=0),
    ">150 µm":    dict(mouse=863774, day=day, dmd=1, roi=1),
}
MAX_TRIALS = 50
# =================================================================

display(
    change_metrics[
        ["subject_id","session_label","session_order","dmd","roi","depth_um","depth_group",
         "change_minus_pre_dff","change_minus_matched_same_image_dff"]
    ].sort_values(["depth_um","subject_id","session_order","dmd","roi"])
)

resolved_change_single_trial_examples = {str(g): dict(CHANGE_EXAMPLES[str(g)]) for g in DEPTH_GROUP_ORDER}
fig,axs=plt.subplots(len(DEPTH_GROUP_ORDER),figsize=(4,6),sharex=True)


for ax,group in zip(np.atleast_1d(axs),DEPTH_GROUP_ORDER):
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    spec=CHANGE_EXAMPLES[str(group)]
    r,s=select_roi_example(spec)
    if str(r["depth_group"]) != str(group):
        warnings.warn(f'{spec} is {r["depth_group"]}, not {group}')

    with h5py.File(s["single_trial_h5"],"r") as h5:
        h5g=h5[f'DMD{int(r["dmd"])}']
        axis=h5_roi_axis(h5g,int(r["dmd"]),int(r["roi"]))
        traces=np.asarray(h5g["change"]["traces"][:,axis,:],float)
        t=reconcile_timebase(h5["timebase_sec/change"][:],traces.shape[-1])

    take=np.arange(len(traces))
    if len(take)>MAX_TRIALS:
        take=np.unique(np.linspace(0,len(traces)-1,MAX_TRIALS).round().astype(int))
    c=DEPTH_COLORS[str(group)]

    for k in take: ax.plot(t,traces[k],color=c,lw=.55,alpha=.22)
    ax.plot(t,np.nanmean(traces[take],axis=0),color="black",lw=1.6)
    ax.axvspan(0,.25,color='lightgray',alpha=.6,lw=0); ax.axvline(0,color=".25",lw=.9,ls="--")
    ax.axvspan(-0.75,-0.5,color='lightgray',alpha=.2,lw=0)
    ax.set(title=f'{group}',
           xlabel="Time from change (s)",ylabel='\u0394F/F$_{0}$')
    finish_axis(ax)


fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,f"change_single_trials_by_depth_{day}"),formats=[".pdf"],dpi=300)
plt.show()

In [ ]:
# ================ OVERLAY CHANGE RESPONSES BY DEPTH ================
SESSION = "A1"
OVERLAY_EXAMPLES = {
    "<100 µm": dict(mouse=852835,dmd=1,roi=0),
    "100–150 µm": dict(mouse=852835,dmd=2,roi=0),
    ">150 µm": dict(mouse=863774,dmd=1,roi=1),
}
SHOW_TRIAL_SEM = True
# ===================================================================

fig,ax=plt.subplots(figsize=(4,3))

for group in DEPTH_GROUP_ORDER:
    ex=OVERLAY_EXAMPLES[str(group)]
    spec=dict(mouse=ex["mouse"],day=SESSION,dmd=ex["dmd"],roi=ex["roi"])
    r,s=select_roi_example(spec)

    if str(r["depth_group"])!=str(group):
        warnings.warn(f'{spec} is {r["depth_group"]}, not {group}')

    with h5py.File(s["single_trial_h5"],"r") as h5:
        h5g=h5[f'DMD{int(r["dmd"])}']
        axis=h5_roi_axis(h5g,int(r["dmd"]),int(r["roi"]))
        traces=np.asarray(h5g["change"]["traces"][:,axis,:],float)
        t=reconcile_timebase(h5["timebase_sec/change"][:],traces.shape[-1])

    # Per-trial correction from stored extraction onset -> true change onset
    idx=indexed_event_table(str(r["session_id"]),int(r["dmd"]),"change")
    if len(idx)!=len(traces):
        raise ValueError(f"Trial-count mismatch: {len(traces)} traces vs {len(idx)} indexed changes")

    offsets=(
        pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()
        - pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy()
    )

    # Re-align each trial so t=0 is the actual change onset
    aligned=np.full_like(traces,np.nan,dtype=float)
    for i,(trace,offset) in enumerate(zip(traces,offsets)):
        aligned[i]=np.interp(t+offset,t,trace,left=np.nan,right=np.nan)

    mean=np.nanmean(aligned,axis=0)
    n=np.sum(np.isfinite(aligned),axis=0)
    sem=np.nanstd(aligned,axis=0,ddof=1)/np.sqrt(n)
    sem[n<2]=np.nan

    c=DEPTH_COLORS[str(group)]
    if SHOW_TRIAL_SEM:
        ax.fill_between(t,mean-sem,mean+sem,color=c,alpha=.20,lw=0)
    ax.plot(t,mean,color=c,lw=1.5,label=str(group))

ax.axvspan(0,.25,color="lightgray",alpha=.6,zorder=0)
ax.axvline(0,color="k",ls="--",zorder=0)
ax.axvspan(-.75,-.5,color="lightgray",alpha=.2,zorder=0)

ax.set(
    xlabel="Time from image change (s)",
    ylabel="ΔF/F$_0$",
    title=f"Single-neuron change responses\nSession {SESSION}"
)
finish_axis(ax)
ax.legend(frameon=False,fontsize=10)
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,f"change_depth_overlay_{SESSION}"),formats=[".pdf"],dpi=300)
plt.show()

## Slide 29 left — Quantification of change window – previous image response

In [ ]:
q=change_metrics.dropna(subset=["pre_change_dff","change_dff"]).copy()
fig,ax=plt.subplots(figsize=(4.4,4.0))
for group,g in q.groupby("depth_group",observed=True):
    ax.scatter(g["pre_change_dff"],g["change_dff"],s=34,color=DEPTH_COLORS[str(group)],alpha=.55,edgecolor="white",linewidth=.3,label=str(group))
vals=np.r_[q["pre_change_dff"],q["change_dff"]]; vals=vals[np.isfinite(vals)]; lo,hi=np.nanmin(vals),np.nanmax(vals); pad=.06*(hi-lo if hi>lo else 1); lim=(lo-pad,hi+pad); ax.plot(lim,lim,color=".55",ls="--",lw=1.2)
ax.set(xlim=lim,ylim=lim,xlabel="Previous-image response\n(−0.75 to −0.50 s; dF/F)",ylabel="Change-window response\n(0 to 0.25 s; dF/F)",title="Change response vs previous image")
add_depth_legend(ax); finish_axis(ax); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"change_window_vs_previous_image"),formats=[".pdf",".png"],dpi=300); plt.show()

## Slide 29 right — Quantification of change modulation – what if there was no change?

In [ ]:
q=change_metrics.dropna(subset=["matched_same_image_dff","change_dff"]).copy()
fig,ax=plt.subplots(figsize=(4.4,4.0))
for group,g in q.groupby("depth_group",observed=True):
    ax.scatter(g["matched_same_image_dff"],g["change_dff"],s=34,color=DEPTH_COLORS[str(group)],alpha=.55,edgecolor="white",linewidth=.3,label=str(group))
vals=np.r_[q["matched_same_image_dff"],q["change_dff"]]; vals=vals[np.isfinite(vals)]; lo,hi=np.nanmin(vals),np.nanmax(vals); pad=.06*(hi-lo if hi>lo else 1); lim=(lo-pad,hi+pad); ax.plot(lim,lim,color=".55",ls="--",lw=1.2)
ax.set(xlim=lim,ylim=lim,xlabel="Matched ordinary same-image response\n(0 to 0.25 s; dF/F)",ylabel="Change response\n(0 to 0.25 s; dF/F)",title="Change-specific modulation")
add_depth_legend(ax); finish_axis(ax); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"change_modulation_matched_same_image"),formats=[".pdf",".png"],dpi=300); plt.show()

## Slide 30 left — Session-wise quantification of image change response relative to previous image

In [ ]:
q=change_metrics.copy()
labels=[s for s in SESSION_ORDER if s in set(q["session_label"].astype(str))]
x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}
fig,ax=plt.subplots(figsize=(6,4))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for group,g in q.groupby("depth_group",observed=True):
    s=(g.groupby("session_label",observed=True)["change_minus_pre_dff"]
       .agg(mean="mean",sem="sem")
       .reindex(labels))
    valid=s["mean"].notna().to_numpy()
    if not valid.any(): continue
    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    ax.fill_between(xx,(s["mean"]-s["sem"]).to_numpy()[valid],(s["mean"]+s["sem"]).to_numpy()[valid],color=c,alpha=.12,lw=0)
    ax.plot(xx,s["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))

if "A2" in xmap and "B0" in xmap: ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
ax.axhline(0,color=".55",lw=1,ls="--")
ax.set(xticks=x,xticklabels=labels,xlabel="Session",ylabel="\u0394 change response\n(post-pre)")
finish_axis(ax); add_depth_legend(ax)

fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"change_minus_previous_image_across_sessions"),formats=[".pdf",".png"],dpi=300)
plt.show()

In [ ]:
# ===== SESSION-TO-SESSION CHANGE IN CHANGE RESPONSE =====
metric="change_minus_pre_dff"
transitions=list(zip(SESSION_ORDER[:-1],SESSION_ORDER[1:]))

q=change_metrics[
    change_metrics["manually_registered"].astype(bool)
    & change_metrics["global_cell_id"].astype(str).ne("")
    & change_metrics["global_cell_id"].astype(str).ne("nan")
].copy()

rows=[]
for prev,curr in transitions:
    a=q[q["session_label"].astype(str).eq(prev)]
    b=q[q["session_label"].astype(str).eq(curr)]

    p=a.merge(
        b,
        on=["subject_id","global_cell_id"],
        suffixes=("_prev","_curr"),
        how="inner",
    )

    for r in p.itertuples(index=False):
        rows.append({
            "subject_id":r.subject_id,
            "global_cell_id":r.global_cell_id,
            "transition":f"{curr} − {prev}",
            "depth_group":r.depth_group_curr,
            "delta_change":getattr(r,f"{metric}_curr")-getattr(r,f"{metric}_prev"),
        })

delta_change=pd.DataFrame(rows)
transition_order=[f"{b} − {a}" for a,b in transitions]
x=np.arange(len(transition_order))

fig,ax=plt.subplots(figsize=(6,4))

for group,g in delta_change.groupby("depth_group",observed=True):
    s=(g.groupby("transition",observed=True)["delta_change"]
       .agg(mean="mean",sem="sem")
       .reindex(transition_order))

    valid=s["mean"].notna().to_numpy()
    if not valid.any(): continue

    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    mean=s["mean"].to_numpy()[valid]
    sem=s["sem"].fillna(0).to_numpy()[valid]

    ax.fill_between(xx,mean-sem,mean+sem,color=c,alpha=.15,lw=0)
    ax.plot(xx,mean,"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))

ax.axhline(0,color=".55",lw=1,ls="--")
ax.axvline(2.5,color=".65",lw=1,ls=":")
ax.set(
    xticks=x,
    xticklabels=transition_order,
    xlabel="Session transition",
    ylabel="Δ change response\n(session-wise)",
)
ax.tick_params(axis="x",labelrotation=35,labelsize=11)
finish_axis(ax)
add_depth_legend(ax)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(SAVE_PATH,"change_response_session_to_session_delta"),
    formats=[".pdf",".png"],
    dpi=300,
)
plt.show()

display(
    delta_change.groupby(["transition","depth_group"],observed=True)["delta_change"]
    .agg(n="count",mean="mean",sem="sem")
    .reindex(transition_order,level="transition")
)

## Within-session change dynamics
Each change is assigned to the first, middle, or final third of the experiment.

In [ ]:
third_x={k:i for i,k in enumerate(WITHIN_SESSION_THIRD_ORDER)}

for metric, metric_label in WITHIN_SESSION_CHANGE_METRICS:
    for mouse in sorted(change_third_metrics["subject_id"].astype(str).unique()):
        fig,axs=plt.subplots(2,3,figsize=(10.4,6.1),sharex=True,sharey=True)
        axs=axs.ravel()

        for ax,label in zip(axs,SESSION_ORDER):
            q=change_third_metrics[
                (change_third_metrics["subject_id"].astype(str)==str(mouse))
                & (change_third_metrics["session_label"].astype(str)==label)
            ].copy()

            # Thin lines = individual neuron trajectories.
            for (_,roi),g in q.groupby(["dmd","roi"],observed=True):
                g=g.assign(x=g["session_third"].astype(str).map(third_x)).dropna(subset=["x"]).sort_values("x")
                if len(g)>1:
                    ax.plot(
                        g["x"],g[metric],
                        color=DEPTH_COLORS[str(g["depth_group"].iloc[0])],
                        lw=.8,alpha=.16,zorder=1
                    )

            # Thick line + SEM = depth-group summary.
            for group,g in q.groupby("depth_group",observed=True):
                s=(g.groupby("session_third",observed=True)[metric]
                     .agg(mean="mean",sem="sem")
                     .reindex(WITHIN_SESSION_THIRD_ORDER))
                valid=s["mean"].notna().to_numpy()
                if not valid.any():
                    continue
                xx=np.arange(3)[valid]
                c=DEPTH_COLORS[str(group)]
                ax.fill_between(
                    xx,(s["mean"]-s["sem"]).to_numpy()[valid],(s["mean"]+s["sem"]).to_numpy()[valid],
                    color=c,alpha=.12,lw=0,zorder=2
                )
                ax.plot(
                    xx,s["mean"].to_numpy()[valid],"-o",
                    color=c,lw=2.5,ms=6,mec="black",mew=.7,zorder=3,label=str(group)
                )

            ax.axhline(0,color=".65",lw=1,ls="--")
            ax.set_title(label)
            finish_axis(ax)

        for ax in axs[3:]:
            ax.set_xticks(range(3),["First","Second","Third"])
            ax.set_xlabel("Experiment third")
        axs[0].set_ylabel(f"{metric_label}\n(ΔdF/F)")
        axs[3].set_ylabel(f"{metric_label}\n(ΔdF/F)")
        add_depth_legend(axs[2])
        fig.suptitle(f"Within-session image-change dynamics · Mouse {mouse}\n{metric_label}",y=.995,fontsize=17)
        fig.tight_layout()

        stem = "change_within_session_thirds_" + (
            "matched_same_image" if "matched" in metric else "previous_image"
        ) + f"_mouse_{mouse}"
        save_figure(fig,os.path.join(SAVE_PATH,stem),formats=[".pdf",".png"],dpi=300)
        plt.show()

## Change-aligned dF/F response shape across neurons

In [ ]:
plot_grid=np.linspace(-1.0,.75,1751)
fig,axs=plt.subplots(2,3,figsize=(11.5,6.0),sharex=True,sharey=True); axs=axs.ravel()
for ax,label in zip(axs,SESSION_ORDER):
    q=change_curve_df[change_curve_df["session_label"].astype(str).eq(label)]
    for group,g in q.groupby("depth_group",observed=True):
        a=np.vstack([interp_trace(row.time,row.mean_dff,plot_grid) for row in g.itertuples(index=False)])
        mean=np.nanmean(a,axis=0); sem=pd.DataFrame(a).sem(axis=0,skipna=True).to_numpy(); c=DEPTH_COLORS[str(group)]
        ax.fill_between(plot_grid,mean-sem,mean+sem,color=c,alpha=.15,lw=0); ax.plot(plot_grid,mean,color=c,lw=2.2,label=str(group))
    ax.axvspan(-.75,-.50,color=".78",alpha=.2,lw=0); ax.axvspan(0,.25,color=".78",alpha=.6,lw=0); ax.axvline(0,color=".35",lw=1,ls="--"); ax.set_title(label); finish_axis(ax)
for ax in axs[3:]: ax.set_xlabel("Time from change (s)")
axs[0].set_ylabel("ΔF/F$_{0}$"); axs[3].set_ylabel("ΔF/F$_{0}$"); add_depth_legend(axs[2])
# fig.suptitle("Change-aligned ΔF/F$_{0}$",y=0.95,fontsize=18); fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"change_dff_session_depth"),formats=[".pdf",'.png'],dpi=300); plt.show()
filen = 'mean_change_response'
save_figure(fig,os.path.join(SAVE_PATH,filen),formats = ['.pdf'],dpi=300)

# 6. Omission responses
Native traces preserve pre-omission dynamics. Ramp slope is quantified from −0.5→+0.5 s.

In [ ]:
omission_rows=[]; omission_event_rows=[]; omission_curve_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); session_rois=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(sid)]
    with h5py.File(session.single_trial_h5,"r") as h5:
        stored_t=np.asarray(h5["timebase_sec/omission"][:],float)
        for r in session_rois.itertuples(index=False):
            group=h5[f"DMD{int(r.dmd)}"]; axis=h5_roi_axis(group,int(r.dmd),int(r.roi)); sub=group["omission"]
            traces=np.asarray(sub["traces"][:,axis,:],float); t=reconcile_timebase(stored_t,traces.shape[-1]); idx=indexed_event_table(sid,int(r.dmd),"omission")
            if len(idx)!=len(traces) or not np.array_equal(idx["trial_index"].astype(int).to_numpy(),np.arange(len(traces))): raise ValueError(f"Omission trial index mismatch for {sid} DMD{int(r.dmd)}")
            offsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()-pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy()
            pre=shifted_window_means(traces,t,offsets,(-.75,-.50)); omission=shifted_window_means(traces,t,offsets,OMISSION_MATCH_WINDOW_S)
            pre_early=shifted_window_means(traces,t,offsets,PRE_OMISSION_RAMP_EARLY_S); pre_late=shifted_window_means(traces,t,offsets,PRE_OMISSION_RAMP_LATE_S)
            ramp_early=shifted_window_means(traces,t,offsets,OMISSION_RAMP_EARLY_S); ramp_late=shifted_window_means(traces,t,offsets,OMISSION_RAMP_LATE_S)
            pre_slope=shifted_window_slopes(traces,t,offsets,PRE_OMISSION_SLOPE_WINDOW_S); omission_only_slope=shifted_window_slopes(traces,t,offsets,(0.0,0.750)); ramp_slope=shifted_window_slopes(traces,t,offsets,OMISSION_RAMP_SLOPE_WINDOW_S)
            post=shifted_window_means(traces,t,offsets,POST_OMISSION_WINDOW_S)
            controls=load_image_control_table(h5,sid,int(r.dmd),int(r.roi),axis)
            matched_cycle=np.array([nearest_control_mean(row.event_image_label,row.cycle_onset_sec,row.sequence_position_expected,controls,"cycle_dff") for row in idx.itertuples(index=False)],float)
            matched_post=np.array([nearest_control_mean(row.next_image_label,row.next_cycle_onset_sec,row.next_sequence_position_expected,controls,"image_dff") if pd.notna(row.next_image_label) and np.isfinite(pd.to_numeric(row.next_cycle_onset_sec,errors="coerce")) else np.nan for row in idx.itertuples(index=False)],float)
            pre_ramp=pre_late-pre_early; full_ramp=ramp_late-ramp_early
            meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
            omission_rows.append({**meta,"pre_omission_dff":float(np.nanmean(pre)),"omission_dff":float(np.nanmean(omission)),"omission_minus_pre_dff":float(np.nanmean(omission-pre)),"matched_cycle_dff":float(np.nanmean(matched_cycle)),"omission_minus_matched_dff":float(np.nanmean(omission-matched_cycle)),"pre_omission_ramp_dff":float(np.nanmean(pre_ramp)),"omission_ramp_dff":float(np.nanmean(full_ramp)),"pre_omission_slope_dff_per_s":float(np.nanmean(pre_slope)),"omission_only_slope_dff_per_s":float(np.nanmean(omission_only_slope)),"ramp_continuation_delta_dff_per_s":float(np.nanmean(omission_only_slope-pre_slope)),"omission_ramp_slope_dff_per_s":float(np.nanmean(ramp_slope)),"post_omission_dff":float(np.nanmean(post)),"matched_post_image_dff":float(np.nanmean(matched_post)),"post_minus_matched_dff":float(np.nanmean(post-matched_post)),"n_omissions":len(traces)})
            omission_event_rows.append(pd.DataFrame({**{k:[v]*len(traces) for k,v in meta.items()},"event_id":idx["event_id"].to_numpy(),"cycle_onset_sec":pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy(),"expected_image_label":idx["event_image_label"].astype(str).to_numpy(),"pre_omission_dff":pre,"omission_dff":omission,"omission_minus_pre_dff":omission-pre,"matched_cycle_dff":matched_cycle,"omission_minus_matched_dff":omission-matched_cycle,"pre_omission_ramp_dff":pre_ramp,"omission_ramp_dff":full_ramp,"pre_omission_slope_dff_per_s":pre_slope,"omission_only_slope_dff_per_s":omission_only_slope,"ramp_continuation_delta_dff_per_s":omission_only_slope-pre_slope,"omission_ramp_slope_dff_per_s":ramp_slope,"post_omission_dff":post,"matched_post_image_dff":matched_post,"post_minus_matched_dff":post-matched_post}))
            omission_curve_rows.append({**meta,"time":t,"mean_dff":np.nanmean(traces,axis=0),"n_omissions":len(traces)})

omission_metrics=pd.DataFrame(omission_rows); omission_event_df=pd.concat(omission_event_rows,ignore_index=True); omission_curve_df=pd.DataFrame(omission_curve_rows)
display(omission_metrics.head())

## Single-trial omission examples by depth

In [ ]:
# =================== SELECT OMISSION EXAMPLES ====================
day = "A2"

OMISSION_EXAMPLES = {
    "<100 µm":    dict(mouse=852835, day=day, dmd=1, roi=0),
    "100–150 µm": dict(mouse=852835, day=day, dmd=2, roi=0),
    ">150 µm":    dict(mouse=863774, day=day, dmd=1, roi=1),
}
MAX_TRIALS = 50
# =================================================================

display(
    omission_metrics[
        ["subject_id","session_label","session_order","dmd","roi","depth_um","depth_group",
         "omission_minus_pre_dff","omission_minus_matched_dff",
         "pre_omission_slope_dff_per_s","omission_only_slope_dff_per_s",
         "omission_ramp_slope_dff_per_s"]
    ].sort_values(["depth_um","subject_id","session_order","dmd","roi"])
)

resolved_omission_single_trial_examples = {str(g):dict(OMISSION_EXAMPLES[str(g)]) for g in DEPTH_GROUP_ORDER}
fig,axs=plt.subplots(len(DEPTH_GROUP_ORDER),figsize=(4,6),sharex=True)

for ax,group in zip(np.atleast_1d(axs),DEPTH_GROUP_ORDER):
    ax.tick_params(axis="x",which="major",reset=True,top=False,labelsize=12)
    ax.tick_params(axis="y",which="major",reset=True,right=False,labelsize=12)

    spec=OMISSION_EXAMPLES[str(group)]
    r,s=select_roi_example(spec)
    if str(r["depth_group"])!=str(group):
        warnings.warn(f'{spec} is {r["depth_group"]}, not {group}')

    with h5py.File(s["single_trial_h5"],"r") as h5:
        h5g=h5[f'DMD{int(r["dmd"])}']
        axis=h5_roi_axis(h5g,int(r["dmd"]),int(r["roi"]))
        traces=np.asarray(h5g["omission"]["traces"][:,axis,:],float)
        t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

    take=np.arange(len(traces))
    if len(take)>MAX_TRIALS:
        take=np.unique(np.linspace(0,len(traces)-1,MAX_TRIALS).round().astype(int))
    c=DEPTH_COLORS[str(group)]

    for k in take: ax.plot(t,traces[k],color=c,lw=.55,alpha=.22)
    ax.plot(t,np.nanmean(traces[take],axis=0),color="black",lw=1.6)

    ax.axvspan(-.75,-.50,color="lightgray",alpha=.4,lw=0)
    ax.axvspan(.75,1.0,color="lightgray",alpha=.4,lw=0)
    ax.axvline(0,color="k",lw=1,ls="--")
    ax.axvline(.25,color="k",lw=1,ls="--")

    ax.set(title=f"{group}",
           xlabel="Time from omission (s)",
           ylabel="ΔF/F$_{0}$")
    finish_axis(ax)

fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,f"omission_single_trials_by_depth_{day}"),formats=[".pdf"],dpi=300)
plt.show()

In [ ]:
# ================= SELECT OMISSION EXAMPLE =================
MOUSE = 852835
SESSION = "B2"
DMD = 1
ROI = 0
MAX_TRIALS = 1
# ===========================================================

spec = dict(mouse=MOUSE, day=SESSION, dmd=DMD, roi=ROI)
r,s = select_roi_example(spec)

with h5py.File(s["single_trial_h5"],"r") as h5:
    h5g = h5[f"DMD{DMD}"]
    axis = h5_roi_axis(h5g,DMD,ROI)
    traces = np.asarray(h5g["omission"]["traces"][:,axis,:],float)
    t = reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

take = np.arange(len(traces))
if len(take)>MAX_TRIALS:
    take = np.unique(np.linspace(0,len(traces)-1,MAX_TRIALS).round().astype(int))

c = DEPTH_COLORS[str(r["depth_group"])]

fig,ax = plt.subplots(figsize=(4,3))
ax.tick_params(axis="x",which="major",reset=True,top=False,labelsize=12)
ax.tick_params(axis="y",which="major",reset=True,right=False,labelsize=12)

for k in take:
    ax.plot(t,traces[k],color=c,lw=.55,alpha=.22)

ax.plot(t,np.nanmean(traces[take],axis=0),color="black",lw=1.6)

ax.axvspan(-.75,-.50,color="lightgray",alpha=.4,lw=0)
ax.axvspan(.75,1.0,color="lightgray",alpha=.4,lw=0)
ax.axvline(0,color="k",lw=1,ls="--")
ax.axvline(.25,color="k",lw=1,ls="--")

ax.set(
    xlabel="Time from omission (s)",
    ylabel="ΔF/F$_0$",
#     title=(
#         f'M{r["subject_id"]} · {r["session_label"]} · '
#         f'DMD{int(r["dmd"])} ROI{int(r["roi"])} · '
#         f'{r["depth_um"]:.0f} µm'
#     )
)

finish_axis(ax)
fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        f'omission_single_trials_M{MOUSE}_{SESSION}_DMD{DMD}_ROI{ROI}'
    ),
    formats=[".pdf"],
    dpi=300
)

plt.show()

In [ ]:
# ================= SELECT OMISSION EXAMPLE =================
MOUSE = 852835
SESSION = "B2"
DMD = 1
ROI = 1

TRIALS = None          # None = evenly sample up to MAX_TRIALS
# TRIALS = [16]    # specific trials
# TRIALS = range(0,11) # trials 0–10

MAX_TRIALS = 50
# ===========================================================

spec=dict(mouse=MOUSE,day=SESSION,dmd=DMD,roi=ROI)
r,s=select_roi_example(spec)

with h5py.File(s["single_trial_h5"],"r") as h5:
    h5g=h5[f"DMD{DMD}"]
    axis=h5_roi_axis(h5g,DMD,ROI)
    traces=np.asarray(h5g["omission"]["traces"][:,axis,:],float)
    t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

# Select trials
if TRIALS is None:
    take=np.arange(len(traces))
    if len(take)>MAX_TRIALS:
        take=np.unique(np.linspace(0,len(traces)-1,MAX_TRIALS).round().astype(int))
else:
    take=np.asarray(list(TRIALS),dtype=int)
    if np.any(take<0) or np.any(take>=len(traces)):
        raise IndexError(f"Valid trial indices are 0–{len(traces)-1}; requested {take.tolist()}")

c=DEPTH_COLORS[str(r["depth_group"])]

fig,ax=plt.subplots(figsize=(5,5))
ax.tick_params(axis="x",which="major",reset=True,top=False,labelsize=12)
ax.tick_params(axis="y",which="major",reset=True,right=False,labelsize=12)

for k in take:
    ax.plot(t,traces[k],color=c,lw=.65,alpha=.28,label=f"Trial {k}" if len(take)<=10 else None)

# Mean only when displaying >1 trial
if len(take)>1:
    ax.plot(t,np.nanmean(traces[take],axis=0),color="black",lw=1.7,label="Mean")

ax.axvspan(-.75,-.50,color="lightgray",alpha=.4,lw=0)
ax.axvspan(.75,1.0,color="lightgray",alpha=.4,lw=0)
ax.axvline(0,color="k",lw=1,ls="--")
ax.axvline(.25,color="k",lw=1,ls="--")

ax.set(
    xlabel="Time from omission (s)",
    ylabel="ΔF/F$_0$",
    title=f'{r["depth_um"]:.0f} µm deep example neuron\nomission response'
)
finish_axis(ax)

if len(take)<=10:
    ax.legend(frameon=False,fontsize=8)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(SAVE_PATH,f"omission_trials_M{MOUSE}_{SESSION}_DMD{DMD}_ROI{ROI}"),
    formats=[".pdf",'.png'],dpi=300
)
plt.show()

print(f"Showing {len(take)}/{len(traces)} omission trials:",take.tolist())

In [ ]:
# ============== STACKED 5-Hz OMISSION TRIALS ==============
MOUSE = 852835
SESSION = "B2"
DMD = 1
ROI = 1

LOWPASS_HZ = 5
FILTER_ORDER = 4
STACK_SPACING = None   # None = determine automatically
# ===========================================================

from scipy import signal

spec=dict(mouse=MOUSE,day=SESSION,dmd=DMD,roi=ROI)
r,s=select_roi_example(spec)

with h5py.File(s["single_trial_h5"],"r") as h5:
    h5g=h5[f"DMD{DMD}"]
    axis=h5_roi_axis(h5g,DMD,ROI)
    traces=np.asarray(h5g["omission"]["traces"][:,axis,:],float)
    t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

# ---------- 5-Hz zero-phase low-pass ----------
fs=1/np.nanmedian(np.diff(t))
sos=signal.butter(FILTER_ORDER,LOWPASS_HZ,btype="lowpass",fs=fs,output="sos")

filtered=np.full_like(traces,np.nan,dtype=float)

for k,y in enumerate(traces):
    good=np.isfinite(y)
    if good.sum()<3: continue

    yfill=np.interp(np.arange(len(y)),np.flatnonzero(good),y[good])
    filtered[k]=signal.sosfiltfilt(sos,yfill)
    filtered[k,~good]=np.nan

# ---------- Choose vertical spacing ----------
if STACK_SPACING is None:
    amp=np.nanpercentile(filtered,95,axis=1)-np.nanpercentile(filtered,5,axis=1)
    STACK_SPACING=1.25*np.nanmedian(amp)

n=len(filtered)

# Trial 0 at top
offsets=(n-1-np.arange(n))*STACK_SPACING

fig,ax=plt.subplots(figsize=(5,max(8,n*.18)))

# Stimulus timing behind traces
ax.axvspan(-.75,-.50,color="lightgray",alpha=.35,lw=0,zorder=0)
ax.axvspan(.75,1.0,color="lightgray",alpha=.35,lw=0,zorder=0)
ax.axvline(0,color=".5",lw=.8,ls="--",zorder=0)
ax.axvline(.25,color=".5",lw=.8,ls="--",zorder=0)

for k,(y,offset) in enumerate(zip(filtered,offsets)):
    ax.plot(t,y+offset,color="black",lw=1.5)

ax.set_yticks(offsets)
ax.set_yticklabels(np.arange(n))
ax.set(
    xlabel="Time from omission (s)",
    ylabel="Omission trial",
    title=f"M{MOUSE} · {SESSION} · DMD{DMD} ROI{ROI}\n5 Hz low-pass"
)

ax.tick_params(axis="y",labelsize=7,length=2)
finish_axis(ax)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        f"omission_5Hz_stacked_M{MOUSE}_{SESSION}_DMD{DMD}_ROI{ROI}"
    ),
    formats=[".pdf"],
    dpi=300
)
plt.show()

In [ ]:
# ============== MULTI-TRIAL OMISSION EXAMPLE ==============
MOUSE = 852835
SESSION = "B2"
DMD = 1
ROI = 1

# TRIALS = [0,3,10,15]     # arbitrary trial indices
TRIALS = range(0,5)    # also works
# ===========================================================

spec=dict(mouse=MOUSE,day=SESSION,dmd=DMD,roi=ROI)
r,s=select_roi_example(spec)

with h5py.File(s["single_trial_h5"],"r") as h5:
    h5g=h5[f"DMD{DMD}"]
    axis=h5_roi_axis(h5g,DMD,ROI)
    traces=np.asarray(h5g["omission"]["traces"][:,axis,:],float)
    t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

take=np.asarray(list(TRIALS),dtype=int)
if np.any(take<0) or np.any(take>=len(traces)):
    raise IndexError(f"Valid trial indices: 0–{len(traces)-1}; requested {take.tolist()}")

colors=plt.cm.tab10(np.linspace(0,1,len(take))) if len(take)<=10 else plt.cm.turbo(np.linspace(0,1,len(take)))

fig,ax=plt.subplots(figsize=(8,6))
ax.tick_params(axis="x",which="major",reset=True,top=False,labelsize=12)
ax.tick_params(axis="y",which="major",reset=True,right=False,labelsize=12)

for k,c in zip(take,colors):
    ax.plot(t,traces[k],color=c,lw=0.75,alpha=1,label=f"Trial {k}")

ax.axvspan(-.75,-.50,color="lightgray",alpha=.4,lw=0,zorder=0)
ax.axvspan(.75,1.0,color="lightgray",alpha=.4,lw=0,zorder=0)
ax.axvline(0,color="k",lw=1,ls="--")
ax.axvline(.25,color="k",lw=1,ls="--")

ax.set(
    xlabel="Time from omission (s)",
    ylabel="ΔF/F$_0$",
    title=f'M{MOUSE} · {SESSION} · DMD{DMD} ROI{ROI}'
)
finish_axis(ax)
# ax.legend(frameon=False,fontsize=8,ncol=1)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(SAVE_PATH,f"omission_selected_trials_M{MOUSE}_{SESSION}_DMD{DMD}_ROI{ROI}"),
    formats=[".pdf"],dpi=300
)
plt.show()

In [ ]:
# ================ INDIVIDUAL OMISSION TRIAL FIGURES ================
MOUSE = 852835
SESSION = "B2"
DMD = 1
ROI = 1

TRIALS = range(20)#[0,3,10]   # any number of trial indices
# ====================================================================

spec=dict(mouse=MOUSE,day=SESSION,dmd=DMD,roi=ROI)
r,s=select_roi_example(spec)

with h5py.File(s["single_trial_h5"],"r") as h5:
    h5g=h5[f"DMD{DMD}"]
    axis=h5_roi_axis(h5g,DMD,ROI)
    traces=np.asarray(h5g["omission"]["traces"][:,axis,:],float)
    t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

take=np.asarray(list(TRIALS),dtype=int)
if np.any(take<0) or np.any(take>=len(traces)):
    raise IndexError(f"Valid trial indices: 0–{len(traces)-1}; requested {take.tolist()}")

for k in take:
    fig,ax=plt.subplots(figsize=(4,3))
    ax.plot(t,traces[k],color="black",lw=1.2)

    ax.axvspan(-.75,-.50,color="lightgray",alpha=.4,lw=0,zorder=0)
    ax.axvspan(.75,1.0,color="lightgray",alpha=.4,lw=0,zorder=0)
    ax.axvline(0,color="k",lw=1,ls="--")
    ax.axvline(.25,color="k",lw=1,ls="--")

    ax.set(
        xlabel="Time from omission (s)",
        ylabel="ΔF/F$_0$",
        title=f"Trial {k}"
    )
    finish_axis(ax)
    fig.tight_layout()

    save_figure(
        fig,
        os.path.join(
            SAVE_PATH,
            f"omission_M{MOUSE}_{SESSION}_DMD{DMD}_ROI{ROI}_trial_{k}"
        ),
        formats=[".pdf"],
        dpi=300
    )
    plt.show()

In [ ]:
# ============== LOW-PASS FILTERED OMISSION TRIALS ==============
MOUSE = 852835
SESSION = "B2"
DMD = 1
ROI = 1

TRIALS = None              # None = all trials
# TRIALS = [0,3,10]
# TRIALS = range(0,10)

LOWPASS_HZ = 20            # low-pass cutoff
FILTER_ORDER = 3
COLOR_MODE = "multicolor"  # "multicolor" or "depth"
TRIAL_ALPHA = .35
# ================================================================

from scipy import signal

spec=dict(mouse=MOUSE,day=SESSION,dmd=DMD,roi=ROI)
r,s=select_roi_example(spec)

with h5py.File(s["single_trial_h5"],"r") as h5:
    h5g=h5[f"DMD{DMD}"]
    axis=h5_roi_axis(h5g,DMD,ROI)
    traces=np.asarray(h5g["omission"]["traces"][:,axis,:],float)
    t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

take=np.arange(len(traces)) if TRIALS is None else np.asarray(list(TRIALS),int)
if np.any(take<0) or np.any(take>=len(traces)):
    raise IndexError(f"Valid trial indices: 0–{len(traces)-1}")

# Zero-phase Butterworth low-pass
fs=1/np.nanmedian(np.diff(t))
sos=signal.butter(FILTER_ORDER,LOWPASS_HZ,btype="lowpass",fs=fs,output="sos")

filtered=[]
for k in take:
    y=traces[k].copy()
    good=np.isfinite(y)
    if good.sum()<3:
        filtered.append(np.full_like(y,np.nan))
        continue
    yfill=np.interp(np.arange(len(y)),np.flatnonzero(good),y[good])
    yf=signal.sosfiltfilt(sos,yfill)
    yf[~good]=np.nan
    filtered.append(yf)

filtered=np.asarray(filtered)
mean=np.nanmean(filtered,axis=0)

# Trial colors
if COLOR_MODE=="multicolor":
    colors=plt.cm.turbo(np.linspace(0,1,len(take)))
elif COLOR_MODE=="depth":
    colors=[DEPTH_COLORS[str(r["depth_group"])]]*len(take)
else:
    raise ValueError("COLOR_MODE must be 'multicolor' or 'depth'")

fig,ax=plt.subplots(figsize=(8,6))

for y,c in zip(filtered,colors):
    ax.plot(t,y,color=c,lw=.8,alpha=TRIAL_ALPHA)

ax.plot(t,mean,color="black",lw=2.2,zorder=5,label="Mean")

ax.axvspan(-.75,-.50,color="lightgray",alpha=.4,lw=0,zorder=0)
ax.axvspan(.75,1.0,color="lightgray",alpha=.4,lw=0,zorder=0)
ax.axvline(0,color="k",lw=1,ls="--")
ax.axvline(.25,color="k",lw=1,ls="--")

ax.set(
    xlabel="Time from omission (s)",
    ylabel="ΔF/F$_0$",
    title=f"M{MOUSE} · {SESSION} · DMD{DMD} ROI{ROI}\n"
          f"{LOWPASS_HZ:g} Hz low-pass · n={len(take)}"
)
finish_axis(ax)
fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        f"omission_lowpass_trials_M{MOUSE}_{SESSION}_DMD{DMD}_ROI{ROI}"
    ),
    formats=[".pdf"],
    dpi=300
)
plt.show()

In [ ]:
# ============== OVERLAY OMISSION RESPONSES BY DEPTH ==============
SESSION = "A1"
OVERLAY_EXAMPLES = {
    "<100 µm": dict(mouse=852835,dmd=1,roi=0),
    "100–150 µm": dict(mouse=852835,dmd=2,roi=0),
    ">150 µm": dict(mouse=863774,dmd=1,roi=1),
}
SHOW_TRIAL_SEM = True
# =================================================================

fig,ax=plt.subplots(figsize=(4,3))

for group in DEPTH_GROUP_ORDER:
    ex=OVERLAY_EXAMPLES[str(group)]
    spec=dict(mouse=ex["mouse"],day=SESSION,dmd=ex["dmd"],roi=ex["roi"])
    r,s=select_roi_example(spec)
    if str(r["depth_group"])!=str(group): warnings.warn(f'{spec} is {r["depth_group"]}, not {group}')

    with h5py.File(s["single_trial_h5"],"r") as h5:
        h5g=h5[f'DMD{int(r["dmd"])}']
        axis=h5_roi_axis(h5g,int(r["dmd"]),int(r["roi"]))
        traces=np.asarray(h5g["omission"]["traces"][:,axis,:],float)
        t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])

    mean=np.nanmean(traces,axis=0)
    sem=pd.DataFrame(traces).sem(axis=0,skipna=True).to_numpy()
    c=DEPTH_COLORS[str(group)]

    if SHOW_TRIAL_SEM: ax.fill_between(t,mean-sem,mean+sem,color=c,alpha=.4,lw=0)
    ax.plot(t,mean,color=c,lw=1.5,label=f'{group}')

ax.axvline(0,color="k",lw=1,ls="--"); ax.axvline(.25,color="k",lw=1,ls='--')
ax.axvspan(-0.75,-0.5,color='lightgray',zorder=0,alpha=0.5)
ax.axvspan(+0.75,+1.0,color='lightgray',zorder=0,alpha=0.5)
ax.set(xlabel="Time from image omission (s)",ylabel="\u0394F/F$_{0}$",
       title=f"Single-neuron omission response\n Session {SESSION}"
      )
finish_axis(ax)
ax.legend(frameon=False,fontsize=8,loc='upper right')
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,f"omission_depth_overlay_{SESSION}"),formats=[".pdf"],dpi=300)
plt.show()

## Slide 32 left — Quantification of omission window – previous image response

In [ ]:
q=omission_metrics.dropna(subset=["pre_omission_dff","omission_dff"]).copy()
fig,ax=plt.subplots(figsize=(4.4,4.0))
for group,g in q.groupby("depth_group",observed=True):
    ax.scatter(g["pre_omission_dff"],g["omission_dff"],s=34,color=DEPTH_COLORS[str(group)],alpha=.55,edgecolor="white",linewidth=.3,label=str(group))
vals=np.r_[q["pre_omission_dff"],q["omission_dff"]]; vals=vals[np.isfinite(vals)]; lo,hi=np.nanmin(vals),np.nanmax(vals); pad=.06*(hi-lo if hi>lo else 1); lim=(lo-pad,hi+pad); ax.plot(lim,lim,color=".55",ls="--",lw=1.2)
ax.set(xlim=lim,ylim=lim,xlabel="Previous-image response\n(−0.75 to −0.50 s; dF/F)",ylabel="Omission-window activity\n(0 to 0.50 s; dF/F)",title="Omission activity vs previous image")
add_depth_legend(ax); finish_axis(ax); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_window_vs_previous_image"),formats=[".pdf",".png"],dpi=300); plt.show()

## Omission ramping across sessions

In [ ]:
q=omission_metrics.copy()
labels=[s for s in SESSION_ORDER if s in set(q["session_label"].astype(str))]
x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}
fig,ax=plt.subplots(figsize=(5,3))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for group,g in q.groupby("depth_group",observed=True):
    s=(g.groupby("session_label",observed=True)["omission_ramp_slope_dff_per_s"]
       .agg(mean="mean",sem="sem")
       .reindex(labels))
    valid=s["mean"].notna().to_numpy()
    if not valid.any(): continue
    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    ax.fill_between(xx,(s["mean"]-s["sem"]).to_numpy()[valid],(s["mean"]+s["sem"]).to_numpy()[valid],color=c,alpha=.12,lw=0)
    ax.plot(xx,s["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))

if "A2" in xmap and "B0" in xmap: ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
ax.axhline(0,color=".55",lw=1,ls="--")
ax.set(xticks=x,xticklabels=labels,xlabel="Session",ylabel="Ramp slope −0.5→+0.5 s (dF/F/s)",title="Omission ramping across sessions")
finish_axis(ax); add_depth_legend(ax)

fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_ramp_slope_across_sessions"),formats=[".pdf",".png"],dpi=300)
plt.show()

## Slide 33 left — Session-wise quantification of omission responses

In [ ]:
q=omission_metrics.copy()
labels=[s for s in SESSION_ORDER if s in set(q["session_label"].astype(str))]
x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}
fig,ax=plt.subplots(figsize=(5,3))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for group,g in q.groupby("depth_group",observed=True):
    s=(g.groupby("session_label",observed=True)["omission_minus_pre_dff"]
       .agg(mean="mean",sem="sem")
       .reindex(labels))
    valid=s["mean"].notna().to_numpy()
    if not valid.any(): continue
    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    ax.fill_between(xx,(s["mean"]-s["sem"]).to_numpy()[valid],(s["mean"]+s["sem"]).to_numpy()[valid],color=c,alpha=.12,lw=0)
    ax.plot(xx,s["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))

if "A2" in xmap and "B0" in xmap: ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
ax.axhline(0,color=".55",lw=1,ls="--")
ax.set(xticks=x,xticklabels=labels,xlabel="Session",ylabel="Omission − pre (\u0394F/F$_{0}$)",title="Omitted image response")
finish_axis(ax); add_depth_legend(ax)

fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_minus_previous_image_across_sessions"),formats=[".pdf",".png"],dpi=300)
plt.show()

In [ ]:
# ========== POST-OMISSION VS PRE-OMISSION IMAGE RESPONSE ==========

fig,ax=plt.subplots(figsize=(5.2,3.7))

plot_longitudinal(
    ax,
    omission_metrics,
    "post_minus_pre_dff",
    "Post − pre image response (ΔF/F$_0$)"
)

ax.axhline(0,color=".55",lw=1,ls="--")
ax.set_title("Post-omission image response")

add_depth_legend(ax)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(SAVE_PATH,"post_vs_pre_omission_image_response"),
    formats=[".pdf",".png"],
    dpi=300
)
plt.show()

## Omission modulation by mouse, session, and depth

In [ ]:
mice=sorted(omission_metrics["subject_id"].astype(str).unique())
fig,axs=plt.subplots(1,len(mice),figsize=(9.8,3.8),sharey=True,squeeze=False); axs=axs.ravel()

for ax,mouse in zip(axs,mice):
    q=omission_metrics[omission_metrics["subject_id"].astype(str)==str(mouse)].copy()
    labels=[s for s in SESSION_ORDER if s in set(q["session_label"].astype(str))]
    x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}

    for group,g in q.groupby("depth_group",observed=True):
        s=(g.groupby("session_label",observed=True)["omission_minus_matched_dff"]
           .agg(mean="mean",sem="sem")
           .reindex(labels))
        valid=s["mean"].notna().to_numpy()
        if not valid.any(): continue
        xx=x[valid]; c=DEPTH_COLORS[str(group)]
        ax.fill_between(xx,(s["mean"]-s["sem"]).to_numpy()[valid],(s["mean"]+s["sem"]).to_numpy()[valid],color=c,alpha=.12,lw=0)
        ax.plot(xx,s["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))

    if "A2" in xmap and "B0" in xmap: ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
    ax.axhline(0,color=".55",lw=1,ls="--")
    ax.set(xticks=x,xticklabels=labels,xlabel="Session",title=f"Mouse {mouse}")
    finish_axis(ax)

axs[0].set_ylabel("Omission − matched ordinary cycle\n(ΔdF/F)")
add_depth_legend(axs[-1])
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_modulation_mouse_session_depth"),formats=[".pdf",".png"],dpi=300)
plt.show()

## Omission-aligned dF/F response shape across neurons

In [ ]:
plot_grid=np.linspace(-1.0,1.25,2251)
fig,axs=plt.subplots(2,3,figsize=(11.5,6.0),sharex=True,sharey=True); axs=axs.ravel()
for ax,label in zip(axs,SESSION_ORDER):
    q=omission_curve_df[omission_curve_df["session_label"].astype(str).eq(label)]
    for group,g in q.groupby("depth_group",observed=True):
        a=np.vstack([interp_trace(row.time,row.mean_dff,plot_grid) for row in g.itertuples(index=False)])
        mean=np.nanmean(a,axis=0); sem=pd.DataFrame(a).sem(axis=0,skipna=True).to_numpy(); c=DEPTH_COLORS[str(group)]
        mean=pd.Series(mean).rolling(10,min_periods=1,center=True).mean().to_numpy(); sem=pd.Series(sem).rolling(10,min_periods=1,center=True).mean().to_numpy()
        ax.fill_between(plot_grid,mean-sem,mean+sem,color=c,alpha=.15,lw=0); ax.plot(plot_grid,mean,color=c,lw=2.2,label=str(group))
    ax.axvspan(-.75,-.50,color="lightgray",alpha=.4,lw=0,zorder=0); ax.axvspan(.75,1.0,color="lightgray",alpha=.4,lw=0,zorder=0)
    ax.axvline(0,color="k",lw=1,ls="--",zorder=0); ax.axvline(0.25,color="k",lw=1,ls="--",zorder=0) 
    ax.set_title(label); finish_axis(ax)
for ax in axs[3:]: ax.set_xlabel("Time from omission (s)")
axs[0].set_ylabel("ΔF/F$_{0}$"); axs[3].set_ylabel("ΔF/F$_{0}$"); add_depth_legend(axs[2])
fig.suptitle("VIP interneuron omission responses across sessions",y=0.975,fontsize=18); fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"omission_dff_session_depth"),formats=[".pdf",'.png'],dpi=300); plt.show()


# 7. Cross-event exploratory analyses

## Omission ramp continuation

In [ ]:
q=omission_metrics.dropna(
    subset=["pre_omission_slope_dff_per_s","omission_only_slope_dff_per_s"]
).copy()

fig,ax=plt.subplots(figsize=(5.0,4.2))
for group,g in q.groupby("depth_group",observed=True):
    ax.scatter(
        g["pre_omission_slope_dff_per_s"],
        g["omission_only_slope_dff_per_s"],
        s=38,color=DEPTH_COLORS[str(group)],alpha=.72,
        edgecolor="black",linewidth=.35,label=str(group)
    )

ax.axhline(0,color=".65",lw=.8)
ax.axvline(0,color=".65",lw=.8)
ax.set(
    xlabel="Pre-omission slope −0.5→0 s (dF/F/s)",
    ylabel="Post-expected-time slope 0→0.5 s (dF/F/s)",
    title="Does pre-omission ramping continue through an omission?",
)
finish_axis(ax)
add_depth_legend(ax)
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_pre_vs_post_ramp_slope"),formats=[".pdf",".png"],dpi=300)
plt.show()

### Ramp components across sessions

In [ ]:
# ================= PRE-OMISSION SLOPE =================
q=omission_metrics.copy()
labels=[s for s in SESSION_ORDER if s in set(q["session_label"].astype(str))]
x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}

fig,ax=plt.subplots(figsize=(5,3))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for group,g in q.groupby("depth_group",observed=True):
    s=(g.groupby("session_label",observed=True)["pre_omission_slope_dff_per_s"]
       .agg(mean="mean",sem="sem")
       .reindex(labels))
    valid=s["mean"].notna().to_numpy()
    if not valid.any(): continue

    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    mean=s["mean"].to_numpy()[valid]
    sem=s["sem"].fillna(0).to_numpy()[valid]

    ax.fill_between(xx,mean-sem,mean+sem,color=c,alpha=.12,lw=0)
    ax.plot(xx,mean,"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))

if "A2" in xmap and "B0" in xmap:
    ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")

ax.axhline(0,color=".55",lw=1,ls="--")
ax.set(
    xticks=x,
    xticklabels=labels,
    xlabel="Session",
    ylabel="Slope (\u0394F/F$_{0}$/s)",
    title="Omission ramping",
)

finish_axis(ax)
add_depth_legend(ax)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(SAVE_PATH,"pre_omission_slope_across_sessions"),
    formats=[".pdf",".png"],
    dpi=300,
)
ax.legend(loc='upper right',frameon=False,fontsize=12)
plt.show()

In [ ]:
# ================= POST-OMISSION SLOPE =================
q=omission_metrics.copy()
labels=[s for s in SESSION_ORDER if s in set(q["session_label"].astype(str))]
x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}

fig,ax=plt.subplots(figsize=(5.2,3.7))

for group,g in q.groupby("depth_group",observed=True):
    s=(g.groupby("session_label",observed=True)["omission_only_slope_dff_per_s"]
       .agg(mean="mean",sem="sem")
       .reindex(labels))
    valid=s["mean"].notna().to_numpy()
    if not valid.any(): continue

    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    mean=s["mean"].to_numpy()[valid]
    sem=s["sem"].fillna(0).to_numpy()[valid]

    ax.fill_between(xx,mean-sem,mean+sem,color=c,alpha=.12,lw=0)
    ax.plot(xx,mean,"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))

if "A2" in xmap and "B0" in xmap:
    ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")

ax.axhline(0,color=".55",lw=1,ls="--")
ax.set(
    xticks=x,
    xticklabels=labels,
    xlabel="Session",
    ylabel="Slope 0→0.75 s (dF/F/s)",
    title="Post-omission ramp",
)

finish_axis(ax)
add_depth_legend(ax)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(SAVE_PATH,"post_omission_slope_across_sessions"),
    formats=[".pdf",".png"],
    dpi=300,
)
plt.show()

### Trial-wise ramp examples

In [ ]:
fig,axs=plt.subplots(1,len(DEPTH_GROUP_ORDER),figsize=(11.5,3.5),sharex=True,sharey=True)
for ax,group in zip(np.atleast_1d(axs),DEPTH_GROUP_ORDER):
    spec=resolved_omission_single_trial_examples[str(group)]
    r,_=select_roi_example(spec)
    q=omission_event_df[
        omission_event_df["session_id"].astype(str).eq(str(r["session_id"]))
        & omission_event_df["dmd"].astype(int).eq(int(r["dmd"]))
        & omission_event_df["roi"].astype(int).eq(int(r["roi"]))
    ].dropna(subset=["pre_omission_slope_dff_per_s","omission_only_slope_dff_per_s"])

    c=DEPTH_COLORS[str(group)]
    ax.scatter(
        q["pre_omission_slope_dff_per_s"],
        q["omission_only_slope_dff_per_s"],
        s=22,color=c,alpha=.45,edgecolor="none"
    )
    ax.scatter(
        [q["pre_omission_slope_dff_per_s"].mean()],
        [q["omission_only_slope_dff_per_s"].mean()],
        s=80,color=c,edgecolor="black",linewidth=.8,zorder=4
    )
    ax.axhline(0,color=".7",lw=.7)
    ax.axvline(0,color=".7",lw=.7)
    ax.set_title(f'{group}\nM{r["subject_id"]} {r["session_label"]} · DMD{int(r["dmd"])} ROI{int(r["roi"])}',fontsize=11)
    ax.set_xlabel("Pre slope (dF/F/s)")
    finish_axis(ax)

axs[0].set_ylabel("0→0.5 s slope (dF/F/s)")
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_trialwise_ramp_continuation_examples"),formats=[".pdf",".png"],dpi=300)
plt.show()

## Omission running-state control
Encoder and event times are compared in session-relative HARP seconds.

In [ ]:
if RUN_RUNNING_CONTROL:
    running_rows = []
    alignment_rows = []

    omission_unique = (
        omission_event_df[["session_id","event_id","cycle_onset_sec"]]
        .drop_duplicates()
        .copy()
    )

    for session in sessions.itertuples(index=False):
        sid = str(session.session_id)
        q = omission_unique[omission_unique["session_id"].astype(str).eq(sid)]
        if q.empty:
            continue

        encoder_path = getattr(session, "encoder_pkl", None)
        if encoder_path is None or not Path(encoder_path).exists():
            warnings.warn(f"{sid}: encoder.pkl unavailable; skipping running-state omission control.")
            continue

        try:
            # encoder.pkl is indexed in absolute HARP time. build_change_detection_events /
            # the extracted-response tables use the same HARP clock after subtracting the
            # session HARP origin. Zeroing the encoder to its first HARP sample therefore
            # places both streams on the same session-relative time axis.
            enc = compute_encoder_velocity(encoder_path, **RUNNING_KWARGS)
        except Exception as exc:
            warnings.warn(f"{sid}: encoder loading failed ({exc}); skipping running-state omission control.")
            continue

        et = np.asarray(enc["time_sec"], float)
        speed = np.asarray(enc["linear_speed_cm_s"], float)
        event_times = pd.to_numeric(q["cycle_onset_sec"], errors="coerce").to_numpy(float)
        event_times = event_times[np.isfinite(event_times)]

        if not len(et) or not len(event_times):
            continue

        alignment_rows.append(dict(
            session_id=sid,
            encoder_start_sec=float(np.nanmin(et)),
            encoder_stop_sec=float(np.nanmax(et)),
            omission_start_sec=float(np.nanmin(event_times)),
            omission_stop_sec=float(np.nanmax(event_times)),
        ))

        overlap = (
            np.nanmax(event_times) >= np.nanmin(et)
            and np.nanmin(event_times) <= np.nanmax(et)
        )
        if not overlap:
            warnings.warn(
                f"{sid}: omission [{np.nanmin(event_times):.2f}, {np.nanmax(event_times):.2f}] s "
                f"does not overlap encoder [{np.nanmin(et):.2f}, {np.nanmax(et):.2f}] s "
                "after session-relative HARP alignment; skipping session."
            )
            continue

        for ev in q.itertuples(index=False):
            onset = float(ev.cycle_onset_sec)
            w = (
                (et >= onset + RUNNING_CLASS_WINDOW_S[0])
                & (et <= onset + RUNNING_CLASS_WINDOW_S[1])
            )
            mean_speed = float(np.nanmean(speed[w])) if w.any() else np.nan
            if not np.isfinite(mean_speed):
                state = np.nan
            elif mean_speed > RUNNING_THRESHOLD_CM_S:
                state = "Running"
            else:
                state = "Stationary"

            running_rows.append(dict(
                session_id=sid,
                event_id=ev.event_id,
                mean_speed_cm_s=mean_speed,
                running_state=state,
            ))

    running_alignment = pd.DataFrame(alignment_rows)
    if len(running_alignment):
        display(running_alignment)

    # Preserve merge keys even when no usable running events are found.
    running_event_state = pd.DataFrame(
        running_rows,
        columns=["session_id","event_id","mean_speed_cm_s","running_state"],
    )

    omission_running_df = omission_event_df.merge(
        running_event_state,
        on=["session_id","event_id"],
        how="left",
        validate="many_to_one",
    )

    omission_running_summary = (
        omission_running_df[
            omission_running_df["running_state"].isin(["Stationary","Running"])
        ]
        .groupby(
            ["subject_id","session_id","session_label","dmd","roi","cell_id","depth_group","running_state"],
            observed=True,
        )
        .agg(
            omission_modulation_dff=("omission_minus_matched_dff","mean"),
            omission_ramp_slope_dff_per_s=("omission_ramp_slope_dff_per_s","mean"),
            n_omissions=("event_id","count"),
        )
        .reset_index()
    )

    if omission_running_summary.empty:
        print("No aligned running/omission events available for plotting.")
    else:
        fig, axs = plt.subplots(1, 2, figsize=(8.2, 3.6))
        metrics = [
            ("omission_modulation_dff", "Omission − matched ordinary dF/F"),
            ("omission_ramp_slope_dff_per_s", "Omission ramp slope (dF/F/s)"),
        ]
        state_x = {"Stationary":0, "Running":1}

        for ax, (metric, ylabel) in zip(axs, metrics):
            q = omission_running_summary.dropna(subset=[metric]).copy()

            for (_, sid, dmd, roi), g in q.groupby(
                ["subject_id","session_id","dmd","roi"], observed=True
            ):
                if g["running_state"].nunique() == 2:
                    g = g.assign(x=g["running_state"].map(state_x)).sort_values("x")
                    ax.plot(g["x"], g[metric], color=".75", lw=.6, alpha=.45, zorder=1)

            for group, g in q.groupby("depth_group", observed=True):
                x = g["running_state"].map(state_x).to_numpy(float)
                jitter = np.linspace(-.045,.045,len(g)) if len(g)>1 else np.array([0.0])
                ax.scatter(
                    x + jitter, g[metric],
                    color=DEPTH_COLORS[str(group)], s=28, alpha=.65,
                    edgecolor="black", linewidth=.3, zorder=2,
                )

            ax.axhline(0, color=".6", lw=.8)
            ax.set(
                xticks=[0,1],
                xticklabels=["Stationary","Running"],
                ylabel=ylabel,
            )
            finish_axis(ax)

        fig.tight_layout()
        save_figure(
            fig,
            os.path.join(SAVE_PATH, "omission_running_state_control"),
            formats=[".pdf"],
            dpi=300,
        )
        plt.show()
else:
    print("Skipping running-state control (RUN_RUNNING_CONTROL=False).")

## Change versus omission phenotype

In [ ]:
phenotype = change_metrics.merge(
    omission_metrics[
        ["session_id","dmd","roi","omission_minus_matched_dff","omission_ramp_slope_dff_per_s"]
    ],
    on=["session_id","dmd","roi"],
    how="inner",
)

fig, ax = plt.subplots(figsize=(5.0, 4.2))
for group, g in phenotype.groupby("depth_group", observed=True):
    ax.scatter(
        g["change_minus_matched_same_image_dff"],
        g["omission_minus_matched_dff"],
        color=DEPTH_COLORS[str(group)], s=38, alpha=.70,
        edgecolor="black", linewidth=.35, label=str(group),
    )
ax.axhline(0, color=".6", lw=.8)
ax.axvline(0, color=".6", lw=.8)
ax.set(
    xlabel="Change − matched same-image dF/F",
    ylabel="Omission − matched ordinary dF/F",
    title="Change and omission modulation in the same neurons",
)
finish_axis(ax)
ax.legend(frameon=False, title="Soma depth", bbox_to_anchor=(1.02,1), loc="upper left")
fig.tight_layout()
save_figure(fig, os.path.join(SAVE_PATH, "change_vs_omission_cell_phenotype"), formats=[".pdf"], dpi=300)
plt.show()

## Expected-image identity during omissions

In [ ]:
omission_identity_rows=[]
group_cols=["subject_id","session_id","session_label","session_order","dmd","roi",
            "cell_id","global_cell_id","manually_registered","depth_um","depth_group"]

for keys,g in omission_event_df.groupby(group_cols,observed=True,dropna=False):
    labels=g["expected_image_label"].astype(str).to_numpy()
    values=pd.to_numeric(g["omission_minus_pre_dff"],errors="coerce").to_numpy(float)
    good=np.isfinite(values); labels,values=labels[good],values[good]
    counts=pd.Series(labels).value_counts()
    keep=set(counts[counts>=MIN_OMISSION_TRIALS_PER_EXPECTED_IMAGE].index.astype(str))
    keep_mask=np.array([x in keep for x in labels],bool); labels,values=labels[keep_mask],values[keep_mask]

    fve=np.nan
    if len(values)>=2 and len(np.unique(labels))>=2:
        grand=float(np.mean(values)); total=float(np.sum((values-grand)**2))
        if total>0:
            stats=pd.DataFrame({"label":labels,"value":values}).groupby("label",observed=True)["value"].agg(["mean","size"])
            between=float(np.sum(stats["size"]*(stats["mean"]-grand)**2))
            fve=between/total

    row=dict(zip(group_cols,keys))
    omission_identity_rows.append({**row,"expected_image_omission_fve":fve,
                                   "n_omission_trials_for_fve":len(values),
                                   "n_expected_images_for_fve":len(np.unique(labels))})

omission_identity_fve=pd.DataFrame(omission_identity_rows)
identity_compare=image_metrics.merge(
    omission_identity_fve[["session_id","dmd","roi","expected_image_omission_fve",
                           "n_omission_trials_for_fve","n_expected_images_for_fve"]],
    on=["session_id","dmd","roi"],how="inner"
)

fig,ax=plt.subplots(figsize=(5,4))
for group,g in identity_compare.groupby("depth_group",observed=True):
    ax.scatter(g["image_fve"],g["expected_image_omission_fve"],color=DEPTH_COLORS[str(group)],
               s=36,alpha=.68,edgecolor="black",linewidth=.35,label=str(group))
ax.axhline(0,color=".65",lw=.8); ax.axvline(0,color=".65",lw=.8)
ax.set(xlabel="Presented-image identity FVE",ylabel="Expected-image identity FVE during omission",
       title="Image identity carried into omission")
finish_axis(ax); ax.legend(frameon=False,title="Soma depth",bbox_to_anchor=(1.02,1),loc="upper left")
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"omission_expected_image_identity_fve"),formats=[".pdf"],dpi=300); plt.show()

## Optional experience-stage summary

In [ ]:
if not EXPERIENCE_STAGE_MAP:
    print(
        "EXPERIENCE_STAGE_MAP is empty. "
        "Fill it in after confirming how A0/A1/A2/B0/B1/B2 map to Familiar/Novel/Novel+."
    )
else:
    ch = change_metrics.copy()
    om = omission_metrics.copy()
    ch["experience_stage"] = ch["session_label"].astype(str).map(EXPERIENCE_STAGE_MAP)
    om["experience_stage"] = om["session_label"].astype(str).map(EXPERIENCE_STAGE_MAP)
    stages = [x for x in EXPERIENCE_STAGE_ORDER if x in set(ch["experience_stage"].dropna()) | set(om["experience_stage"].dropna())]
    xmap = {x:i for i,x in enumerate(stages)}

    fig, axs = plt.subplots(1, 3, figsize=(11.0, 3.5))
    panels = [
        (axs[0], ch, "change_minus_matched_same_image_dff", "Change − matched dF/F"),
        (axs[1], om, "omission_minus_matched_dff", "Omission − matched dF/F"),
        (axs[2], om, "omission_ramp_slope_dff_per_s", "Omission ramp slope (dF/F/s)"),
    ]
    for ax, q, metric, ylabel in panels:
        q = q[q["experience_stage"].isin(stages)].dropna(subset=[metric]).copy()
        for group, g in q.groupby("depth_group", observed=True):
            summary = g.groupby("experience_stage", observed=True)[metric].median().reindex(stages)
            valid = summary.notna().to_numpy()
            ax.plot(
                np.arange(len(stages))[valid],
                summary.to_numpy()[valid],
                "-o",
                color=DEPTH_COLORS[str(group)],
                lw=2.2, ms=6, mec="black", mew=.5,
                label=str(group),
            )
        ax.axhline(0, color=".65", lw=.8)
        ax.set(xticks=np.arange(len(stages)), xticklabels=stages, ylabel=ylabel)
        finish_axis(ax)

    axs[-1].legend(frameon=False, title="Soma depth", bbox_to_anchor=(1.02,1), loc="upper left")
    fig.tight_layout()
    save_figure(fig, os.path.join(SAVE_PATH, "change_omission_experience_stage_summary"), formats=[".pdf"], dpi=300)
    plt.show()

## Composite summary figure for image variability slide

Three-panel summary: one example neuron's image-wise mean dF/F responses, aggregate total response variability by depth, and image-identity FVE across sessions.

In [ ]:
SUMMARY_FIG_EXAMPLE = dict(mouse=863774, day="A2", dmd=1, roi=0)
SUMMARY_FIG_SAVE_NAME = "image_variability_summary_figure"

image_colors = [
    "#c5cae9", "#ffcdd2", "#c8e6c9", "#ffe0b2",
    "#e1bee7", "#d7ccc8", "#cfd8dc", "#b2ebf2",
]

# ---------- Example neuron: image-wise mean responses ----------
r, s = select_roi_example(SUMMARY_FIG_EXAMPLE)
pkg = load_response_package(s["mean_npz"])
key = f"DMD{int(r['dmd'])}"

image_curves = []
if key not in pkg:
    raise KeyError(f"{key} not present in {s['mean_npz']}")

for image in pkg[key].get("image_identity", {}):
    try:
        t, y = get_mean_response(
            pkg,
            dmd=int(r["dmd"]),
            source_roi=int(r["roi"]),
            event_type="image",
            image_name=image,
        )
    except (KeyError, IndexError, ValueError):
        continue

    t = np.asarray(t, float).reshape(-1)
    y = np.asarray(y, float).squeeze()

    if y.ndim != 1 or len(y) != len(t):
        continue

    yy = interp_trace(
        t,
        y,
        np.linspace(
            LATENCY_PLOT_WINDOW_S[0],
            LATENCY_PLOT_WINDOW_S[1],
            751,
        ),
    )

    image_curves.append((
        str(image),
        yy,
        np.nanmean(
            yy[
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) >= IMAGE_WINDOW_S[0])
                &
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) < IMAGE_WINDOW_S[1])
            ]
        )
        -
        np.nanmean(
            yy[
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) >= IMAGE_BASELINE_S[0])
                &
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) < IMAGE_BASELINE_S[1])
            ]
        ),
    ))

if not image_curves:
    raise RuntimeError(
        f"No image-wise mean responses found for {SUMMARY_FIG_EXAMPLE}"
    )

example_grid = np.linspace(
    LATENCY_PLOT_WINDOW_S[0],
    LATENCY_PLOT_WINDOW_S[1],
    751,
)

# Keep the order deterministic and assign one of the eight requested colors.
image_curves = sorted(image_curves, key=lambda x: x[0])

# ---------- Figure layout ----------
# Top row: example image-wise responses + total variability.
# Bottom row: FVE across sessions spanning the full width.
fig = plt.figure(figsize=(6.5, 5.5))
gs = fig.add_gridspec(
    2, 2,
    height_ratios=[1.0, 0.95],
    hspace=0.42,
    wspace=0.34,
)

ax_example = fig.add_subplot(gs[0, 0])
ax_var = fig.add_subplot(gs[0, 1])
ax_fve = fig.add_subplot(gs[1, :])

# ---------- Example neuron: image-wise mean responses ----------
for i, (image, y, magnitude) in enumerate(image_curves):
    ax_example.plot(
        example_grid,
        y,
        color=image_colors[i % len(image_colors)],
        lw=1.5,
        alpha=0.95,
        label=Path(image.replace("\\", "/")).stem,
    )

ax_example.axvspan(
    0,
    IMAGE_WINDOW_S[1],
    color="0.75",
    alpha=0.10,
    lw=0,
)
ax_example.axvline(
    0,
    color="0.45",
    lw=1,
    ls="--",
)

ax_example.set_title(
    "Single-neuron image-wise\nmean responses",fontsize=15
)
ax_example.set_xlabel(
    "Time from image onset (s)",fontsize=12
)
ax_example.set_ylabel(
    "Mean \u0394F/F$_{0}$",fontsize=12
)

finish_axis(ax_example)

# ax_example.text(
#     0.02,
#     0.98,
#     (
#         f'Mouse {r["subject_id"]} · {r["session_label"]}\n'
#         f'DMD{int(r["dmd"])} ROI{int(r["roi"])} · {r["depth_group"]}'
#     ),
#     transform=ax_example.transAxes,
#     ha="left",
#     va="top",
#     fontsize=7.5,
#     color="0.25",
# )

# ---------- Aggregate total response variability ----------
order = [
    g for g in DEPTH_GROUP_ORDER
    if g in set(cell_metrics["depth_group"].astype(str))
]

sns.violinplot(
    data=cell_metrics,
    x="depth_group",
    y="total_dff_variance",
    order=order,
    palette=DEPTH_COLORS,
    inner=None,
    width=0.5,
    linewidth=1.2,
    ax=ax_var,
)

for coll in ax_var.collections:
    coll.set_alpha(0.7)
    coll.set_edgecolor("black")

rng = np.random.default_rng(8)
xpos = (
    np.array(
        [order.index(str(x)) for x in cell_metrics["depth_group"]],
        float,
    )
    + rng.uniform(-0.12, 0.12, len(cell_metrics))
)

ax_var.scatter(
    xpos,
    cell_metrics["total_dff_variance"],
    s=28,
    c=[
        DEPTH_COLORS[str(x)]
        for x in cell_metrics["depth_group"]
    ],
    edgecolor="black",
    linewidth=0.45,
    zorder=3,
    alpha=0.85,
)

ax_var.set_title(
    "Total trial-wise\nresponse variability",fontsize=15
)
ax_var.set_xlabel(
    "Depth bin",fontsize=12
)
ax_var.set_ylabel(
    "Variance",fontsize=12
)

finish_axis(ax_var)

# ---------- FVE across sessions ----------
labels=[s for s in SESSION_ORDER if s in set(image_metrics["session_label"].astype(str))]
x=np.arange(len(labels)); xmap={s:i for i,s in enumerate(labels)}
for group,g in image_metrics.groupby("depth_group",observed=True):
    q=(g.groupby("session_label",observed=True)["image_fve"]
       .agg(mean="mean",sem="sem")
       .reindex(labels))
    valid=q["mean"].notna().to_numpy()
    if not valid.any(): continue
    xx=x[valid]; c=DEPTH_COLORS[str(group)]
    ax_fve.fill_between(xx,(q["mean"]-q["sem"]).to_numpy()[valid],(q["mean"]+q["sem"]).to_numpy()[valid],color=c,alpha=.12,lw=0)
    ax_fve.plot(xx,q["mean"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group))
if "A2" in xmap and "B0" in xmap: ax_fve.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
ax_fve.set(xticks=x,xticklabels=labels,xlabel="Session",ylabel="FVE")
finish_axis(ax_fve)

ax_fve.axhline(
    0,
    color="0.55",
    lw=1,
    ls="--",
)

ax_fve.set_title(
    "FVE over sessions",fontsize=15
)
add_depth_legend(
    ax_fve,
    loc="upper right",
)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        SUMMARY_FIG_SAVE_NAME,
    ),
    formats=[".pdf",'.png'],
    dpi=300,
)

plt.show()